**GUIDE**

The goal of `fastllm` library which we are currently refactoring module by module here in these dialogs is to create a unified llm api for OpenAI responses, OpenAI chat, Anthropic and Gemini apis. Users should be able to use this unified / canonical library to be able to swap models/providers effortlessly without needing to change any code.

Refactored dialogs so far:

- `aai-ws/fastllm/nbs/00_types` : Defines the core canonical types: `Part`/`PartType` (atomic content units), `Msg` (conversation turns), and `RequestOptions` (unified generation parameters). `Caps` was removed. Includes caching notes per provider.

- `aai-ws/fastllm/nbs/01_normalize` : Contains: `ToolCall` dataclass + per-provider tool call extraction (anthropic, openai responses, openai chat, gemini), `Usage` dataclass + per-provider usage normalization, `Completion` dataclass, `canon_finish` for finish reason mapping, `model_and_provider` helper, and full `normalize_*` functions for all 4 endpoints (openai responses, openai chat, anthropic messages, gemini generate).

- `aai-ws/fastllm/nbs/02_streaming` : Contains `Delta` (streaming event fragment), `PartAccum` (part accumulator for collation), generic `_acollect_stream`, per-provider streaming normalizers and index functions (openai responses, openai chat, anthropic, gemini), and the unified `acollect_stream` dispatcher. Tests cover text, thinking, tool calls, and web search across all providers. Uses Kimi for OpenAI Chat thinking tests.

- `aai-ws/fastllm/nbs/03_acomplete` : Unified `acomplete` function that denorms canonical `Msg`/`Part` back to provider-specific payloads and calls APIs. Contains: per-provider `denorm_*_msgs` (text, thinking, tool_use, tool_result, server tools), `denorm_*_tools`, `denorm_*_tool_choice`, `denorm_*_reasoning`, media denorm (images, audio, video, files), Anthropic `cache_control` passthrough, `mk_client`/`infer_api` for auto provider detection + third-party support (Kimi, DeepSeek, etc.), system message handling, and `acollect_stream` integration. All-to-all provider swap tests (60 combos) passing. Also fixed `server=tc.server` propagation into `Part.data` across `01_normalize` and `02_streaming`.

You can find the pre-refactor and fully functional snapshot of this library `~/aai-ws/fastllm/fastllm_py/*.py`. The refactored and nbdev exported code exists in `~/aai-ws/fastllm/fastllm/*.py`. The already refactored modules might differ from their counterparts in, as refactoring involves simplifying code, refactoring the code to fastai / literate programming style, redesigning parts, and fixing issues/bugs.


Please go over and compare these refactored dialogs with their original versions in `fastllm_py/`, and summarize the refactoring process, which we'll use a mental model going forward.

Re 1 we already start from a copied version of the original .py in this dialog, and we'll be actively refactoring it here

Please bring over any TODOs / gotchas or important message from the previous dialog that we'll be using while working on this dialog

# Clients

> Provider clients built on the OpenAPI operation layer.

In [ ]:
#| default_exp acomplete

## Imports

In [ ]:
#| export
import nbdev, yaml, json
from dataclasses import dataclass, field, fields
from fastcore.utils import *
from fastcore.meta import *
from fastspec.spec import *
from fastspec.oapi import *

from fastllm.types import *
from fastllm.normalize import *
from fastllm.streaming import *

In [ ]:
#| hide
import yaml,json,base64
from fastcore.test import *
from cachy.core import enable_cachy, disable_cachy, doms

In [ ]:
enable_cachy(doms=(*doms,'api.moonshot.ai'), hdrs=('content-type',))

In [ ]:
from lisette.core import lite_mk_func

In [ ]:
#| export
repo_path = nbdev.config.get_config().config_path; repo_path
specs_path = repo_path/'specs'

ant_spec  = SpecParser.from_openapi(dict2obj(yaml.safe_load((specs_path/'anthropic.yml').read_text())))
oai_spec  = SpecParser.from_openapi(dict2obj(yaml.safe_load((specs_path/'openai.with-code-samples.yml').read_text())))
gem_spec  = SpecParser.from_discovery(dict2obj(json.loads((specs_path/'gemini.json').read_text())))

In [ ]:
kimi_cli = OpenAPIClient(oai_spec, headers={"Authorization": f"Bearer {os.environ['MOONSHOT_API_KEY']}"})
for op in kimi_cli.ops: op.base_url = 'https://api.moonshot.ai/v1'

In [ ]:
ant_spec, oai_spec, gem_spec

(SpecParser(base_url='https://api.anthropic.com', ops=47),
 SpecParser(base_url='https://api.openai.com/v1', ops=241),
 SpecParser(base_url='https://generativelanguage.googleapis.com/', ops=79))

In [ ]:
ant_cli = OpenAPIClient(ant_spec, headers={"x-api-key": os.environ["ANTHROPIC_API_KEY"], "anthropic-version": "2023-06-01"})
oai_cli = OpenAPIClient(oai_spec, headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"})
gem_cli = OpenAPIClient(gem_spec, headers={"x-goog-api-key": os.environ["GEMINI_API_KEY"]})

In [ ]:
ant_cli.groups.keys()

dict_keys(['messages', 'complete', 'models', 'files', 'skills', 'messages_beta_true', 'models_beta_true', 'files_beta_true', 'skills_beta_true'])

In [ ]:
oai_cli.groups.keys()

dict_keys(['assistants', 'audio', 'batches', 'chat', 'completions', 'containers', 'conversations', 'embeddings', 'evals', 'files', 'fine_tuning', 'images', 'models', 'moderations', 'organization', 'projects', 'realtime', 'responses', 'threads', 'uploads', 'vector_stores', 'videos', 'skills', 'chatkit'])

In [ ]:
gem_cli.groups.keys()

dict_keys(['batches', 'models', 'tuned_models', 'dynamic', 'cached_contents', 'media', 'files', 'generated_files', 'file_search_stores', 'corpora'])

In [ ]:
gem_cli.models

- models.generate_content(model, contents, system_instruction, tools, tool_config, safety_settings, generation_config, cached_content, service_tier, store): *Generates a model response given an input `GenerateContentRequest`. Refer to the [text generation guide](https://ai.google.dev/gemini-api/docs/text-generation) for detailed usage information. Input capabilities differ between models, including tuned models. Refer to the [model guide](https://ai.google.dev/gemini-api/docs/models/gemini) and [tuning guide](https://ai.google.dev/gemini-api/docs/model-tuning) for details.*
- models.generate_answer(model, contents, answer_style, inline_passages, semantic_retriever, safety_settings, temperature): *Generates a grounded answer from the model given an input `GenerateAnswerRequest`.*
- models.stream_generate_content(model, contents, system_instruction, tools, tool_config, safety_settings, generation_config, cached_content, service_tier, store): *Generates a [streamed response](https://ai.google.dev/gemini-api/docs/text-generation?lang=python#generate-a-text-stream) from the model given an input `GenerateContentRequest`.*
- models.embed_content(model, content, task_type, title, output_dimensionality): *Generates a text embedding vector from the input `Content` using the specified [Gemini Embedding model](https://ai.google.dev/gemini-api/docs/models/gemini#text-embedding).*
- models.batch_embed_contents(model, requests): *Generates multiple embedding vectors from the input `Content` which consists of a batch of strings represented as `EmbedContentRequest` objects.*
- models.count_tokens(model, contents, generate_content_request): *Runs a model's tokenizer on input `Content` and returns the token count. Refer to the [tokens guide](https://ai.google.dev/gemini-api/docs/tokens) to learn more about tokens.*
- models.batch_generate_content(model, batch): *Enqueues a batch of `GenerateContent` requests for batch processing.*
- models.async_batch_embed_content(model, batch): *Enqueues a batch of `EmbedContent` requests for batch processing. We have a `BatchEmbedContents` handler in `GenerativeService`, but it was synchronized. So we name this one to be `Async` to avoid confusion.*
- models.generate_message(model, prompt, temperature, candidate_count, top_p, top_k): *Generates a response from the model given an input `MessagePrompt`.*
- models.count_message_tokens(model, prompt): *Runs a model's tokenizer on a string and returns the token count.*
- models.get(name): *Gets information about a specific `Model` such as its version number, token limits, [parameters](https://ai.google.dev/gemini-api/docs/models/generative-models#model-parameters) and other metadata. Refer to the [Gemini models guide](https://ai.google.dev/gemini-api/docs/models/gemini) for detailed model information.*
- models.list(page_size, page_token): *Lists the [`Model`s](https://ai.google.dev/gemini-api/docs/models/gemini) available through the Gemini API.*
- models.predict(model, instances, parameters): *Performs a prediction request.*
- models.predict_long_running(model, instances, parameters): *Same as Predict but returns an LRO.*
- models.generate_text(model, prompt, temperature, candidate_count, max_output_tokens, top_p, top_k, safety_settings, stop_sequences): *Generates a response from the model given an input message.*
- models.embed_text(model, text): *Generates an embedding from the model given an input message.*
- models.batch_embed_text(model, texts, requests): *Generates multiple embeddings from the model given input text in a synchronous call.*
- models.count_text_tokens(model, prompt): *Runs a model's tokenizer on a text and returns the token count.*

In [ ]:
import json
from lisette.core import lite_mk_func

In [ ]:
def simple_add(a:int,b:int):
    'add numbers'
    return a + b

In [ ]:
sch = lite_mk_func(simple_add);sch

{'type': 'function',
 'function': {'name': 'simple_add',
  'description': 'add numbers',
  'parameters': {'type': 'object',
   'properties': {'a': {'description': '', 'type': 'integer'},
    'b': {'description': '', 'type': 'integer'}},
   'required': ['a', 'b']}}}

## Refactor

Let's start with a highlevel summary of what this module does to get a better idea for refactoring, and deciding what's need keeping/dropping or changing

### OpenAI Responses Denorm Msg

Converting our internal represenations Msg/Part back to provider specific payload is the right idea. In `normalize` and `streaming` we convert api responses to internal representation, and here we translate back. Ok let's start with tool call: 

In [ ]:
mn,inp = 'gpt-4o-mini','What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.'
tools=[{"type": "function", **sch['function']}]
resp = await oai_cli.responses.create_response(model=mn, input=inp, tools=tools, stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai): pass

In [ ]:
resp_comp.message.content

[Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'fc_0637c35cbc783b2e0069d7bad21bb48193b1e426c977e52b5a', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'server': False, 'type': 'function_call', 'status': 'completed', 'call_id': 'call_dKULgc7ZwOThLi6YVhw2CMEv'}),
 Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'fc_0637c35cbc783b2e0069d7bad21bc48193a6668686b57b1802', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 20}, 'server': False, 'type': 'function_call', 'status': 'completed', 'call_id': 'call_xldlxaK8iqg5vE8DRXGCRJEX'})]

In [ ]:
mn,msgs = 'gpt-4o-mini', [{"role": "user", "content": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}]
resp = await oai_cli.chat.create_chat_completion(model=mn,messages=msgs, tools=[sch], stream=True, stream_options={"include_usage": True})
async for chat_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai_chat): pass

In [ ]:
chat_comp.message.content

[Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'call_lH6XatachWXyyOIOHeSMr6Mh', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'server': False, 'index': 0, 'type': 'function'}),
 Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'call_ZMbiQm8Hah7xrNCNnUhUP9w5', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 20}, 'server': False, 'index': 1, 'type': 'function'})]

In [ ]:
mn,msgs = 'claude-sonnet-4-20250514', [{"role": "user", "content": "What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel."}]
think,hdrs = {"type": "enabled", "budget_tokens": 5000}, {"anthropic-version": "2023-06-01"}
tools = [{"name": "simple_add", "description": "add numbers", "input_schema": sch['function']['parameters']}]
resp = await ant_cli.messages.messages_post(model=mn, messages=msgs, tools=tools, max_tokens=8000, thinking=think, headers=hdrs, stream=True)
async for ant_comp in acollect_stream(resp,model=mn,api_name=ApiName.anthropic): pass

In [ ]:
ant_comp.message.content

[Part(type=<PartType.thinking: 'thinking'>, text="The user is asking for two addition operations:\n1. 3 + 5\n2. 10 + 20\n\nThey specifically want me to use the simple_add tool and do it in parallel. I have the simple_add function available which takes two integer parameters 'a' and 'b'. I can make both function calls in the same function_calls block to execute them in parallel.\n\nFor the first calculation: a=3, b=5\nFor the second calculation: a=10, b=20", data=None),
 Part(type=<PartType.text: 'text'>, text="I'll calculate both additions for you using the simple_add tool in parallel.", data=None),
 Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'toolu_016rVuo7JtGaDnRPWvvmRAKb', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'server': False, 'caller': {'type': 'direct'}}),
 Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'toolu_01Jpvs7DMGusSkYqxmE59dsF', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 20}, 'server': False, 'caller': {'type'

In [ ]:
mn,inp = 'models/gemini-3-flash-preview', [{"role": "user", "parts": [{"text": 'What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.'}]}]
resp = await gem_cli.models.stream_generate_content(model=mn, contents=inp, tools=[{"functionDeclarations": [sch['function']]}], stream=True, _query={"alt": "sse"}) # required for streaming: stream=True, _query={"alt": "sse"}
async for gem_comp in acollect_stream(resp,model=mn,api_name=ApiName.gemini): pass

In [ ]:
gem_comp.message.content

[Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'vchmcu53', 'name': 'simple_add', 'arguments': {'b': 5, 'a': 3}, 'server': False, 'thoughtSignature': 'EqUDCqIDAb4+9vvgiBWJe8rHQ4AsefEidQLNf3wESYqUssWiAripZiGOGQlKRycajyxuPB7kSU1sCX6zg/ULNAjMY9qklM9iUN38J2LqKwgUjJ6JC1grDOj+VxBT4DuB+2NZ1BKk5+pxMSp9/1vkArpfMN8ekYx8nk4izcpODjj846vfl8sKFu0sMHjDazItyTUrfPFniUVcX8aGbdMVjQQt5jh9DcFI5nBh8LD1hrlck29IRvwnL0OcbyndUDyKewMS8msM22dsYE4gJg0AqrLcCsKBb+arW9zLWQFcC85qEtCLs+EQMdk6pTSYQOrZf2KJZJHXW3kv11ioq8p81aSyq6xnAUSDs9og7azl5XKyL+nWXe5Zum3aCyjva4NEDpIotMvrW0wH1cXPaR7WPqxVg9qk/YNuGgRlZ2cQDrKLuge1/z9/vk9v4w4/t24sJzYJRdFH4RWgdTqmHXhFz78Q62hAQtHXHpWXnwHtrQ/yRyzdimy3DC2XHt9SbEDd4ULaJy/AI0EyyDJTFfZALh2F5tyLf7iK929aVChkAUUg6p3AsR95XQ=='}),
 Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': '7rwckpni', 'name': 'simple_add', 'arguments': {'b': 20, 'a': 10}, 'server': False})]

Here we see examples of tool calls from each provider returning our IR, we probably want a way to convert `tool_use` parts into provider specific parts that will be used to create the input/msgs/contents to be passed back to create_response/create_chat_completion/messages_post/generate_content. wdyt

> Looking at the tool_use Part data across providers, there's a common core (id, name, arguments) plus provider-specific extras:

BTW you don't need to and shouldn't infer things like this yourself, always look at the source dialog

Ok let's start with responses api's tool denorm func, have a look at the schema and write `denorm_openai_tool_use` func

In [ ]:
#| export
def denorm_openai_responses_tool_use(p:Part):
    "Convert canonical tool_use Part back to OpenAI Responses function_call item."
    return dict(type='function_call', call_id=p.data.get('call_id') or p.data.get('id', ''), name=p.data.get('name', ''), arguments=json.dumps(p.data.get('arguments', '{}')))

In [ ]:
@patch(as_prop=True)
def tool_use_parts(self:Completion): return L(self.message.content).filter(lambda o: o.type==PartType.tool_use)

In [ ]:
resp_comp.tool_use_parts.map(denorm_openai_responses_tool_use)

[{'type': 'function_call', 'call_id': 'call_dKULgc7ZwOThLi6YVhw2CMEv', 'name': 'simple_add', 'arguments': '{"a": 3, "b": 5}'}, {'type': 'function_call', 'call_id': 'call_xldlxaK8iqg5vE8DRXGCRJEX', 'name': 'simple_add', 'arguments': '{"a": 10, "b": 20}'}]

Ok next step is the tool results, how will users pass that with IR?

have a look at the previous dialogs to find out, we have `PartType.tool_result`

In [ ]:
#| export
def mk_tool_res_msg(tool_parts:Part, results:str):
    'A user util'
    return Msg(role="tool", content=[Part(type=PartType.tool_result, text=res, data=p.data) for p,res in zip(tool_parts, results)])

In [ ]:
tparts = resp_comp.tool_use_parts
tool_res_parts = L(mk_tool_res_msg(tparts, (8,30)).content)
tool_res_parts

[Part(type=<PartType.tool_result: 'tool_result'>, text=8, data={'id': 'fc_0637c35cbc783b2e0069d7bad21bb48193b1e426c977e52b5a', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'server': False, 'type': 'function_call', 'status': 'completed', 'call_id': 'call_dKULgc7ZwOThLi6YVhw2CMEv'}), Part(type=<PartType.tool_result: 'tool_result'>, text=30, data={'id': 'fc_0637c35cbc783b2e0069d7bad21bc48193a6668686b57b1802', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 20}, 'server': False, 'type': 'function_call', 'status': 'completed', 'call_id': 'call_xldlxaK8iqg5vE8DRXGCRJEX'})]

In [ ]:
#| export
def denorm_openai_responses_tool_result(m:Part):
    "Convert canonical tool result back to OpenAI Responses function_call_output item."
    return dict(type='function_call_output', call_id=m.data.get('call_id') or m.data.get('id', ''), output=str(m.text))

In [ ]:
tool_res_parts.map(denorm_openai_responses_tool_result)

[{'type': 'function_call_output', 'call_id': 'call_dKULgc7ZwOThLi6YVhw2CMEv', 'output': '8'}, {'type': 'function_call_output', 'call_id': 'call_xldlxaK8iqg5vE8DRXGCRJEX', 'output': '30'}]

Ok what's needed next to send back the tool results with their response

In [ ]:
resp_comp.tool_use_parts

[Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'fc_0637c35cbc783b2e0069d7bad21bb48193b1e426c977e52b5a', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'server': False, 'type': 'function_call', 'status': 'completed', 'call_id': 'call_dKULgc7ZwOThLi6YVhw2CMEv'}), Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'fc_0637c35cbc783b2e0069d7bad21bc48193a6668686b57b1802', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 20}, 'server': False, 'type': 'function_call', 'status': 'completed', 'call_id': 'call_xldlxaK8iqg5vE8DRXGCRJEX'})]

In [ ]:
mn,inp = 'gpt-4o-mini','What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.'
input_items = [
    {"type": "message", "role": "user", "content": [{"type": "input_text", "text": inp}]},
    *resp_comp.tool_use_parts.map(denorm_openai_responses_tool_use),
    *tool_res_parts.map(denorm_openai_responses_tool_result),
]
input_items

[{'type': 'message',
  'role': 'user',
  'content': [{'type': 'input_text',
    'text': 'What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.'}]},
 {'type': 'function_call',
  'call_id': 'call_dKULgc7ZwOThLi6YVhw2CMEv',
  'name': 'simple_add',
  'arguments': '{"a": 3, "b": 5}'},
 {'type': 'function_call',
  'call_id': 'call_xldlxaK8iqg5vE8DRXGCRJEX',
  'name': 'simple_add',
  'arguments': '{"a": 10, "b": 20}'},
 {'type': 'function_call_output',
  'call_id': 'call_dKULgc7ZwOThLi6YVhw2CMEv',
  'output': '8'},
 {'type': 'function_call_output',
  'call_id': 'call_xldlxaK8iqg5vE8DRXGCRJEX',
  'output': '30'}]

are you sure this is the right format? So shouldn't function calls have an assistant role and similarly for tool outputs?

Refer to the schema and online docs pls

In [ ]:
mn,inp = 'gpt-4o-mini','What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.'
input_items = [
    {"type": "message", "role": "user", "content": [{"type": "input_text", "text": inp}]},
    *resp_comp.tool_use_parts.map(denorm_openai_responses_tool_use),
    *tool_res_parts.map(denorm_openai_responses_tool_result),
]

tools=[{"type": "function", **sch['function']}]
resp = await oai_cli.responses.create_response(model=mn, input=input_items, tools=tools, stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai): pass
print(resp_comp.message.content[0].text)

The results are as follows:

- \(3 + 5 = 8\)
- \(10 + 20 = 30\)


In [ ]:
msg1 = Msg(role='user', content=[Part(type=PartType.text, text='What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.')])

Maybe before that we also need a `denorm_openai_responses_user`

Yes let's start with text

In [ ]:
def denorm_openai_responses_user(m:Msg):
    "Convert canonical user Msg to OpenAI Responses input message."
    parts = [{"type": "input_text", "text": p.text or ""} for p in m.content if p.type == PartType.text]
    return {"type": "message", "role": "user", "content": parts}

In [ ]:
inp = denorm_openai_responses_user(msg1); inp

{'type': 'message',
 'role': 'user',
 'content': [{'type': 'input_text',
   'text': 'What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.'}]}

In [ ]:
tools=[{"type": "function", **sch['function']}]
resp = await oai_cli.responses.create_response(model=mn, input=[inp], tools=tools, stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai): pass
resp_comp

Completion(model='gpt-4o-mini-2024-07-18', message=Msg(role='assistant', content=[Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'fc_0615a6f278d65be30069df900c7bbc819093074395a21a964a', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'server': False, 'type': 'function_call', 'status': 'completed', 'call_id': 'call_68clwdKpkiJz5p5nh00FepRq'}), Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'fc_0615a6f278d65be30069df900c7bcc819087f7fd3efb06cad2', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 20}, 'server': False, 'type': 'function_call', 'status': 'completed', 'call_id': 'call_AmmaMYv5TMZnbITz4V6j99P1'})], data=None), finish_reason=<finish_reason.tool_calls: 'tool_calls'>, usage=Usage(prompt_tokens=60, completion_tokens=53, total_tokens=113, raw={'input_tokens': 60, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 53, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 113}), tool_calls=[ToolCall(id='fc_0615a6f278

In [ ]:
msg2 = resp_comp.message

In [ ]:
#| export
def denorm_openai_responses_assistant(m:Msg):
    items, srv_call_id = [], None
    for p in m.content:
        if p.type == PartType.tool_use:
            items.append(denorm_openai_responses_tool_use(p))
            srv_call_id = (p.data.get('call_id') or p.data.get('id','')) if p.data.get('server') else None
        elif p.type in (PartType.text, PartType.thinking) and srv_call_id:
            items.append(dict(type='function_call_output', call_id=srv_call_id, output=p.text or ''))
            srv_call_id = None
        elif p.type in (PartType.text, PartType.thinking):
            items.append(dict(type="message", role="assistant", content=[dict(type="output_text", text=p.text)]))
        ... # TODO: potentially other parts
    return items

In [ ]:
denorm_openai_responses_assistant(msg2)

[{'type': 'function_call',
  'call_id': 'call_68clwdKpkiJz5p5nh00FepRq',
  'name': 'simple_add',
  'arguments': '{"a": 3, "b": 5}'},
 {'type': 'function_call',
  'call_id': 'call_AmmaMYv5TMZnbITz4V6j99P1',
  'name': 'simple_add',
  'arguments': '{"a": 10, "b": 20}'}]

In [ ]:
msg3 = mk_tool_res_msg(resp_comp.tool_use_parts, ('8', '30')); msg3

Msg(role='tool', content=[Part(type=<PartType.tool_result: 'tool_result'>, text='8', data={'id': 'fc_0615a6f278d65be30069df900c7bbc819093074395a21a964a', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'server': False, 'type': 'function_call', 'status': 'completed', 'call_id': 'call_68clwdKpkiJz5p5nh00FepRq'}), Part(type=<PartType.tool_result: 'tool_result'>, text='30', data={'id': 'fc_0615a6f278d65be30069df900c7bcc819087f7fd3efb06cad2', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 20}, 'server': False, 'type': 'function_call', 'status': 'completed', 'call_id': 'call_AmmaMYv5TMZnbITz4V6j99P1'})], data=None)

In [ ]:
#| export
def denorm_openai_responses_tool(m:Msg):
    items = []
    for part in m.content:
        if part.type == PartType.tool_result: items.append(denorm_openai_responses_tool_result(part))
        ... # TODO: potentially other parts
    return items

In [ ]:
denorm_openai_responses_tool(msg3)

[{'type': 'function_call_output',
  'call_id': 'call_68clwdKpkiJz5p5nh00FepRq',
  'output': '8'},
 {'type': 'function_call_output',
  'call_id': 'call_AmmaMYv5TMZnbITz4V6j99P1',
  'output': '30'}]

In [ ]:
msgs = [msg1, msg2, msg3]; msgs

[Msg(role='user', content=[Part(type=<PartType.text: 'text'>, text='What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.', data=None)], data=None),
 Msg(role='assistant', content=[Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'fc_0615a6f278d65be30069df900c7bbc819093074395a21a964a', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'server': False, 'type': 'function_call', 'status': 'completed', 'call_id': 'call_68clwdKpkiJz5p5nh00FepRq'}), Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'fc_0615a6f278d65be30069df900c7bcc819087f7fd3efb06cad2', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 20}, 'server': False, 'type': 'function_call', 'status': 'completed', 'call_id': 'call_AmmaMYv5TMZnbITz4V6j99P1'})], data=None),
 Msg(role='tool', content=[Part(type=<PartType.tool_result: 'tool_result'>, text='8', data={'id': 'fc_0615a6f278d65be30069df900c7bbc819093074395a21a964a', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'server': Fa

"message": "Invalid value: 'function_call'. Supported values are: 'input_text', 'input_image', 'output_text', 'refusal', 'input_file', 'computer_screenshot', and 'summary_text'.",

In [ ]:
#| export
def denorm_openai_responses_msgs(msgs:list[Msg]):
    res = []
    for m in msgs:
        if m.role == 'user':        res.append(denorm_openai_responses_user(m))
        elif m.role == 'assistant': res.extend(denorm_openai_responses_assistant(m))
        elif m.role == 'tool':      res.extend(denorm_openai_responses_tool(m))
    return res

In [ ]:
input_items = denorm_openai_responses_msgs(msgs); input_items

[{'type': 'message',
  'role': 'user',
  'content': [{'type': 'input_text',
    'text': 'What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.'}]},
 {'type': 'function_call',
  'call_id': 'call_68clwdKpkiJz5p5nh00FepRq',
  'name': 'simple_add',
  'arguments': '{"a": 3, "b": 5}'},
 {'type': 'function_call',
  'call_id': 'call_AmmaMYv5TMZnbITz4V6j99P1',
  'name': 'simple_add',
  'arguments': '{"a": 10, "b": 20}'},
 {'type': 'function_call_output',
  'call_id': 'call_68clwdKpkiJz5p5nh00FepRq',
  'output': '8'},
 {'type': 'function_call_output',
  'call_id': 'call_AmmaMYv5TMZnbITz4V6j99P1',
  'output': '30'}]

In [ ]:
resp = await oai_cli.responses.create_response(model=mn, input=input_items, tools=tools, stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai): pass
print(resp_comp.message.content[0].text)

None


In [ ]:
msg1 = Msg(role='user', content=[Part(type=PartType.text, text='Hi!')])
resp = await oai_cli.responses.create_response(model=mn, input=denorm_openai_responses_msgs([msg1]), stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai): pass
print(resp_comp.message.content[0].text)

Hello! How can I assist you today?


In [ ]:
msg2,msg3 = resp_comp.message, Msg(role='user', content=[Part(type=PartType.text, text='How are you?')])
resp = await oai_cli.responses.create_response(model=mn, input=denorm_openai_responses_msgs([msg1,msg2,msg3]), stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai): pass
print(resp_comp.message.content[0].text)

I’m here and ready to help! How about you?


In [ ]:
denorm_openai_responses_msgs([msg1,msg2,msg3])

[{'type': 'message',
  'role': 'user',
  'content': [{'type': 'input_text', 'text': 'Hi!'}]},
 {'type': 'message',
  'role': 'assistant',
  'content': [{'type': 'output_text',
    'text': 'Hello! How can I assist you today?'}]},
 {'type': 'message',
  'role': 'user',
  'content': [{'type': 'input_text', 'text': 'How are you?'}]}]

Nice we got a set of working `denorm` functions which can be used from IR to openai responses payload, of course we'll update these functions to support other things as we go. Shall we a look at openai chat now unless you have a better idea?

Yes please check the schema and write similar denorm functions for openai chat as above

In [ ]:
mn,inp = 'o4-mini','What is 31231231 * 12312?'
msg1 = Msg(role='user', content=[Part(type=PartType.text, text=inp)])
resp = await oai_cli.responses.create_response(model=mn, input=denorm_openai_responses_msgs([msg1]), reasoning={"effort": "low", "summary": "auto"}, stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai): pass
print(resp_comp.message.content)

[Part(type=<PartType.thinking: 'thinking'>, text="**Calculating the product**\n\nThe user wants to compute 31,231,231 multiplied by 12,312. I can break this down for clarity.\n\nFirst, I'll calculate each part separately: multiplying by 12,000 and 312. For the first part, multiplying 31,231,231 by 12,000 gives 374,774,772,000. For the second part, multiplying by 312 results in 9,744,144,072.\n\nSumming these components together, I get 384,518,916,072. To confirm, I can redo the calculations using a standard method or a calculator.", data={'id': 'rs_03d72b4e32ab972a0069df90149750819fa8fca250587f623d', 'type': 'reasoning', 'summary': [{'type': 'summary_text', 'text': "**Calculating the product**\n\nThe user wants to compute 31,231,231 multiplied by 12,312. I can break this down for clarity.\n\nFirst, I'll calculate each part separately: multiplying by 12,000 and 312. For the first part, multiplying 31,231,231 by 12,000 gives 374,774,772,000. For the second part, multiplying by 312 result

In [ ]:
msg2,msg3 = resp_comp.message, Msg(role='user', content=[Part(type=PartType.text, text='Great!')])
resp = await oai_cli.responses.create_response(model=mn, input=denorm_openai_responses_msgs([msg1,msg2,msg3]), reasoning={"effort": "low"}, stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai): pass
print(resp_comp.message.content)

[Part(type=<PartType.text: 'text'>, text='I’m glad I could help! Let me know if there’s anything else you need.', data={'type': 'output_text', 'annotations': [], 'logprobs': [], 'text': 'I’m glad I could help! Let me know if there’s anything else you need.'})]


In [ ]:
denorm_openai_responses_msgs([msg1,msg2,msg3])

[{'type': 'message',
  'role': 'user',
  'content': [{'type': 'input_text', 'text': 'What is 31231231 * 12312?'}]},
 {'type': 'message',
  'role': 'assistant',
  'content': [{'type': 'output_text',
    'text': "**Calculating the product**\n\nThe user wants to compute 31,231,231 multiplied by 12,312. I can break this down for clarity.\n\nFirst, I'll calculate each part separately: multiplying by 12,000 and 312. For the first part, multiplying 31,231,231 by 12,000 gives 374,774,772,000. For the second part, multiplying by 312 results in 9,744,144,072.\n\nSumming these components together, I get 384,518,916,072. To confirm, I can redo the calculations using a standard method or a calculator."}]},
 {'type': 'message',
  'role': 'assistant',
  'content': [{'type': 'output_text',
    'text': "**Verifying calculations**\n\nI’m breaking down the number 12,312 and confirming it's equal to 2 + 10 + 300 + 12,000. That matches my previous calculations of 12,000 + 312. \n\nWhen I re-evaluate 31,231

In [ ]:
msg1 = Msg(role='user', content=[Part(type=PartType.text, text='What is the weather in Istanbul today?')])
tools=[{"type": "web_search_preview"}]
resp = await oai_cli.responses.create_response(model=mn, input=denorm_openai_responses_msgs([msg1]), tools=tools, stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai): pass
print(resp_comp.message.content)

[Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'ws_0f1f64839cc104e60069df90252660819c8f83bcb8d3ea14eb', 'name': 'web_search', 'arguments': {'type': 'search', 'queries': ['weather: Istanbul, Turkey'], 'query': 'weather: Istanbul, Turkey'}, 'server': True, 'type': 'web_search_call', 'status': 'completed', 'action': {'type': 'search', 'queries': ['weather: Istanbul, Turkey'], 'query': 'weather: Istanbul, Turkey'}}), Part(type=<PartType.text: 'text'>, text='Today in Istanbul (Thursday, April 15, 2026), the weather is as follows:\n\n• Current conditions: Mostly sunny, 62°F (16°C)   \n• Early evening (around 5 PM): Cloudy, 61°F (16°C)   \n• Later tonight (10 PM): Cloudy, 52°F (11°C)   \n\nYou can expect mostly cloudy skies through the night, with temperatures gradually falling into the low 50s°F (around 10–12°C).', data={'type': 'output_text', 'annotations': [], 'logprobs': [], 'text': 'Today in Istanbul (Thursday, April 15, 2026), the weather is as follows:\n\n• Current 

In [ ]:
msg2,msg3 = resp_comp.message, Msg(role='user', content=[Part(type=PartType.text, text='Great!')])
resp = await oai_cli.responses.create_response(model=mn, input=denorm_openai_responses_msgs([msg1,msg2,msg3]), stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai): pass
print(resp_comp.message.content)

[Part(type=<PartType.text: 'text'>, text='Glad to help! Let me know if there’s anything else you need.', data={'type': 'output_text', 'annotations': [], 'logprobs': [], 'text': 'Glad to help! Let me know if there’s anything else you need.'})]


In [ ]:
resp_comp.tool_calls

[]

In [ ]:
resp_comp.message.content[0]

Part(type=<PartType.text: 'text'>, text='Glad to help! Let me know if there’s anything else you need.', data={'type': 'output_text', 'annotations': [], 'logprobs': [], 'text': 'Glad to help! Let me know if there’s anything else you need.'})

`server=True` tool use needs special handling, but looks like `server=` from ToolCall is not included in Part.tool_use, can you check normalize and streaming dialog to see if this is the case for all providers?

Let's fix it with (1)

Yes, and check if a similar fix is needed in streaming 

In [ ]:
denorm_openai_responses_msgs([resp_comp.message])

[{'type': 'message',
  'role': 'assistant',
  'content': [{'type': 'output_text',
    'text': 'Glad to help! Let me know if there’s anything else you need.'}]}]

Ok for messages with server tool use let's use the following parts as its result and fix it in `denorm_openai_responses_msgs`

To keep the all to all conversion relationship across providers we can't rely on data? The following text should become the tool result iiuc?

Yes fix the code

In [ ]:
denorm_openai_responses_msgs([resp_comp.message])

[{'type': 'message',
  'role': 'assistant',
  'content': [{'type': 'output_text',
    'text': 'Glad to help! Let me know if there’s anything else you need.'}]}]

No what I meant to do was, we'll add the `denorm_openai_responses_tool_use(p)` as a tool use as before, and also add the remaining text/thinking as tool result if server=True

In [ ]:
denorm_openai_responses_msgs([resp_comp.message])

[{'type': 'message',
  'role': 'assistant',
  'content': [{'type': 'output_text',
    'text': 'Glad to help! Let me know if there’s anything else you need.'}]}]

In [ ]:
msg1 = Msg(role='user', content=[Part(type=PartType.text, text='What is the weather in Istanbul today?')])
tools=[{"type": "web_search_preview"}]
resp = await oai_cli.responses.create_response(model=mn, input=denorm_openai_responses_msgs([msg1]), tools=tools, stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai): pass
print(resp_comp.message.content)

[Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'ws_0f1f64839cc104e60069df90252660819c8f83bcb8d3ea14eb', 'name': 'web_search', 'arguments': {'type': 'search', 'queries': ['weather: Istanbul, Turkey'], 'query': 'weather: Istanbul, Turkey'}, 'server': True, 'type': 'web_search_call', 'status': 'completed', 'action': {'type': 'search', 'queries': ['weather: Istanbul, Turkey'], 'query': 'weather: Istanbul, Turkey'}}), Part(type=<PartType.text: 'text'>, text='Today in Istanbul (Thursday, April 15, 2026), the weather is as follows:\n\n• Current conditions: Mostly sunny, 62°F (16°C)   \n• Early evening (around 5 PM): Cloudy, 61°F (16°C)   \n• Later tonight (10 PM): Cloudy, 52°F (11°C)   \n\nYou can expect mostly cloudy skies through the night, with temperatures gradually falling into the low 50s°F (around 10–12°C).', data={'type': 'output_text', 'annotations': [], 'logprobs': [], 'text': 'Today in Istanbul (Thursday, April 15, 2026), the weather is as follows:\n\n• Current 

In [ ]:
msg2,msg3 = resp_comp.message, Msg(role='user', content=[Part(type=PartType.text, text='Great!')])
resp = await oai_cli.responses.create_response(model=mn, input=denorm_openai_responses_msgs([msg1,msg2,msg3]), stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai): pass
print(resp_comp.message.content)

[Part(type=<PartType.text: 'text'>, text='Glad to help! Let me know if there’s anything else you need.', data={'type': 'output_text', 'annotations': [], 'logprobs': [], 'text': 'Glad to help! Let me know if there’s anything else you need.'})]


#### Text:

Demonstrating how canonical back and forth provider translations work

In [ ]:
mn = 'gpt-4o-mini'

In [ ]:
msg1 = Msg(role='user', content=[Part(type=PartType.text, text='Hi!')])
resp = await oai_cli.responses.create_response(model=mn, input=denorm_openai_responses_msgs([msg1]), stream=True)
async for comp in acollect_stream(resp, model=mn, api_name=ApiName.openai): pass
print(comp.message.content[0].text)

Hello! How can I assist you today?


In [ ]:
denorm_openai_responses_msgs([msg1])

[{'type': 'message',
  'role': 'user',
  'content': [{'type': 'input_text', 'text': 'Hi!'}]}]

In [ ]:
msg2,msg3 = comp.message, Msg(role='user', content=[Part(type=PartType.text, text='How are you?')])
resp = await oai_cli.responses.create_response(model=mn, input=denorm_openai_responses_msgs([msg1,msg2,msg3]), stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai): pass
print(resp_comp.message.content[0].text)

I’m here and ready to help! How about you?


In [ ]:
denorm_openai_responses_msgs([msg1,msg2,msg3])

[{'type': 'message',
  'role': 'user',
  'content': [{'type': 'input_text', 'text': 'Hi!'}]},
 {'type': 'message',
  'role': 'assistant',
  'content': [{'type': 'output_text',
    'text': 'Hello! How can I assist you today?'}]},
 {'type': 'message',
  'role': 'user',
  'content': [{'type': 'input_text', 'text': 'How are you?'}]}]

#### Thinking

In [ ]:
mn,inp = 'o4-mini','What is 31231231 * 12312?'
msg1 = Msg(role='user', content=[Part(type=PartType.text, text=inp)])
resp = await oai_cli.responses.create_response(model=mn, input=denorm_openai_responses_msgs([msg1]), reasoning={"effort": "low", "summary": "auto"}, stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai): pass
L(resp_comp.message.content).attrgot('type')

[<PartType.thinking: 'thinking'>, <PartType.thinking: 'thinking'>, <PartType.text: 'text'>]

In [ ]:
denorm_openai_responses_msgs([msg1])

[{'type': 'message',
  'role': 'user',
  'content': [{'type': 'input_text', 'text': 'What is 31231231 * 12312?'}]}]

In [ ]:
msg2,msg3 = resp_comp.message, Msg(role='user', content=[Part(type=PartType.text, text='Great!')])
resp = await oai_cli.responses.create_response(model=mn, input=denorm_openai_responses_msgs([msg1,msg2,msg3]), reasoning={"effort": "low"}, stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai): pass
print(resp_comp.message.content[0].text)

I’m glad I could help! Let me know if there’s anything else you need.


In [ ]:
denorm_openai_responses_msgs([msg1,msg2,msg3])

[{'type': 'message',
  'role': 'user',
  'content': [{'type': 'input_text', 'text': 'What is 31231231 * 12312?'}]},
 {'type': 'message',
  'role': 'assistant',
  'content': [{'type': 'output_text',
    'text': "**Calculating the product**\n\nThe user wants to compute 31,231,231 multiplied by 12,312. I can break this down for clarity.\n\nFirst, I'll calculate each part separately: multiplying by 12,000 and 312. For the first part, multiplying 31,231,231 by 12,000 gives 374,774,772,000. For the second part, multiplying by 312 results in 9,744,144,072.\n\nSumming these components together, I get 384,518,916,072. To confirm, I can redo the calculations using a standard method or a calculator."}]},
 {'type': 'message',
  'role': 'assistant',
  'content': [{'type': 'output_text',
    'text': "**Verifying calculations**\n\nI’m breaking down the number 12,312 and confirming it's equal to 2 + 10 + 300 + 12,000. That matches my previous calculations of 12,000 + 312. \n\nWhen I re-evaluate 31,231

#### Tool Call

In [ ]:
mn,inp = 'gpt-4o-mini','What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.'
tools=[{"type": "function", **sch['function']}]
msg1 = Msg(role='user', content=[Part(type=PartType.text, text=inp)])
resp = await oai_cli.responses.create_response(model=mn, input=denorm_openai_responses_msgs([msg1]), tools=tools, stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai): pass
L(resp_comp.message.content).attrgot('type')

[<PartType.tool_use: 'tool_use'>, <PartType.tool_use: 'tool_use'>]

In [ ]:
denorm_openai_responses_msgs([msg1])

[{'type': 'message',
  'role': 'user',
  'content': [{'type': 'input_text',
    'text': 'What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.'}]}]

In [ ]:
tmsg = mk_tool_res_msg(resp_comp.tool_use_parts, (8,30)); tmsg

Msg(role='tool', content=[Part(type=<PartType.tool_result: 'tool_result'>, text=8, data={'id': 'fc_0615a6f278d65be30069df900c7bbc819093074395a21a964a', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'server': False, 'type': 'function_call', 'status': 'completed', 'call_id': 'call_68clwdKpkiJz5p5nh00FepRq'}), Part(type=<PartType.tool_result: 'tool_result'>, text=30, data={'id': 'fc_0615a6f278d65be30069df900c7bcc819087f7fd3efb06cad2', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 20}, 'server': False, 'type': 'function_call', 'status': 'completed', 'call_id': 'call_AmmaMYv5TMZnbITz4V6j99P1'})], data=None)

In [ ]:
msg2 = resp_comp.message
resp = await oai_cli.responses.create_response(model=mn, input=denorm_openai_responses_msgs([msg1,msg2,tmsg]), stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai): pass
print(resp_comp.message.content[0].text)

The results are: 

- \(3 + 5 = 8\)
- \(10 + 20 = 30\)


In [ ]:
denorm_openai_responses_msgs([msg1,msg2,tmsg])

[{'type': 'message',
  'role': 'user',
  'content': [{'type': 'input_text',
    'text': 'What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.'}]},
 {'type': 'function_call',
  'call_id': 'call_68clwdKpkiJz5p5nh00FepRq',
  'name': 'simple_add',
  'arguments': '{"a": 3, "b": 5}'},
 {'type': 'function_call',
  'call_id': 'call_AmmaMYv5TMZnbITz4V6j99P1',
  'name': 'simple_add',
  'arguments': '{"a": 10, "b": 20}'},
 {'type': 'function_call_output',
  'call_id': 'call_68clwdKpkiJz5p5nh00FepRq',
  'output': '8'},
 {'type': 'function_call_output',
  'call_id': 'call_AmmaMYv5TMZnbITz4V6j99P1',
  'output': '30'}]

#### Web Search

In [ ]:
msg1 = Msg(role='user', content=[Part(type=PartType.text, text='What is the weather in Istanbul today?')])
tools=[{"type": "web_search_preview"}]
resp = await oai_cli.responses.create_response(model=mn, input=denorm_openai_responses_msgs([msg1]), tools=tools, stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai): pass
print(resp_comp.message.content)

[Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'ws_013e6c4a274c4e460069df902f6084819587eea1ecbac8fba0', 'name': 'web_search', 'arguments': {'type': 'search', 'queries': ['Istanbul weather today'], 'query': 'Istanbul weather today'}, 'server': True, 'type': 'web_search_call', 'status': 'completed', 'action': {'type': 'search', 'queries': ['Istanbul weather today'], 'query': 'Istanbul weather today'}}), Part(type=<PartType.text: 'text'>, text="As of 4:18 PM local time in Istanbul, the weather is mostly sunny with a temperature of 60°F (15°C).\n\n## Weather for Hastane, Hadımköy-İstanbul Caddesi 34, 34555 Arnavutköy/İstanbul, Türkiye:\nCurrent Conditions: Mostly sunny, 60°F (15°C)\n\nHourly Forecast:\n* 5:00\u202fPM: 56°F (14°C), Cloudy\n* 6:00\u202fPM: 53°F (12°C), Cloudy\n* 7:00\u202fPM: 54°F (12°C), Partly sunny\n* 8:00\u202fPM: 51°F (10°C), Partly cloudy\n* 9:00\u202fPM: 50°F (10°C), Intermittent clouds\n* 10:00\u202fPM: 49°F (10°C), Intermittent clouds\n* 11:00\u2

In [ ]:
denorm_openai_responses_msgs([msg1])

[{'type': 'message',
  'role': 'user',
  'content': [{'type': 'input_text',
    'text': 'What is the weather in Istanbul today?'}]}]

In [ ]:
msg2,msg3 = resp_comp.message, Msg(role='user', content=[Part(type=PartType.text, text='Great!')])
resp = await oai_cli.responses.create_response(model=mn, input=denorm_openai_responses_msgs([msg1,msg2,msg3]), stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai): pass
print(resp_comp.message.content[0].text)

I'm glad you found it helpful! If you have any more questions about the weather or anything else, feel free to ask.


In [ ]:
denorm_openai_responses_msgs([msg1,msg2,msg3])

[{'type': 'message',
  'role': 'user',
  'content': [{'type': 'input_text',
    'text': 'What is the weather in Istanbul today?'}]},
 {'type': 'function_call',
  'call_id': 'ws_013e6c4a274c4e460069df902f6084819587eea1ecbac8fba0',
  'name': 'web_search',
  'arguments': '{"type": "search", "queries": ["Istanbul weather today"], "query": "Istanbul weather today"}'},
 {'type': 'function_call_output',
  'call_id': 'ws_013e6c4a274c4e460069df902f6084819587eea1ecbac8fba0',
  'output': "As of 4:18 PM local time in Istanbul, the weather is mostly sunny with a temperature of 60°F (15°C).\n\n## Weather for Hastane, Hadımköy-İstanbul Caddesi 34, 34555 Arnavutköy/İstanbul, Türkiye:\nCurrent Conditions: Mostly sunny, 60°F (15°C)\n\nHourly Forecast:\n* 5:00\u202fPM: 56°F (14°C), Cloudy\n* 6:00\u202fPM: 53°F (12°C), Cloudy\n* 7:00\u202fPM: 54°F (12°C), Partly sunny\n* 8:00\u202fPM: 51°F (10°C), Partly cloudy\n* 9:00\u202fPM: 50°F (10°C), Intermittent clouds\n* 10:00\u202fPM: 49°F (10°C), Intermittent c

### OpenAI Chat Denorm

In [ ]:
#| export
def denorm_openai_chat_tool_use(p:Part):
    "Convert canonical tool_use Part to OpenAI Chat tool_call dict."
    return dict(id=p.data.get('id') or p.data.get('call_id', ''), type='function',
                function=dict(name=p.data.get('name', ''), arguments=json.dumps(p.data.get('arguments', '{}'))))

def denorm_openai_chat_tool_result(p:Part):
    "Convert canonical tool_result Part to OpenAI Chat tool message."
    return dict(role='tool', tool_call_id=p.data.get('id') or p.data.get('call_id', ''), content=str(p.text))

def denorm_openai_chat_user(m:Msg):
    "Convert canonical user Msg to OpenAI Chat user message."
    if len(m.content) == 1 and m.content[0].type == PartType.text: return dict(role='user', content=m.content[0].text or '')
    return dict(role='user', content=[dict(type='text', text=p.text or '') for p in m.content if p.type == PartType.text])

def denorm_openai_chat_assistant(m:Msg):
    "Convert canonical assistant Msg to OpenAI Chat assistant message + synthetic tool responses for server tools."
    tcs = [denorm_openai_chat_tool_use(p) for p in m.content if p.type == PartType.tool_use]
    msg, srv_responses, non_srv_texts, srv_call_id = dict(role='assistant'), [], [], None
    for p in m.content:
        if p.type == PartType.tool_use:
            srv_call_id = (p.data.get('id') or p.data.get('call_id','')) if p.data.get('server') else None
        elif p.type in (PartType.text, PartType.thinking) and srv_call_id:
            srv_responses.append(dict(role='tool', tool_call_id=srv_call_id, content=p.text or ''))
            srv_call_id = None
        elif p.type == PartType.text: non_srv_texts.append(p)
    if non_srv_texts: msg['content'] = non_srv_texts[0].text if len(non_srv_texts)==1 else [dict(type='text', text=p.text or '') for p in non_srv_texts]
    if tcs: msg['tool_calls'] = tcs
    thinking = [p for p in m.content if p.type == PartType.thinking]
    if thinking: msg['reasoning_content'] = ''.join(p.text or '' for p in thinking)
    return [msg] + srv_responses

def denorm_openai_chat_tool(m:Msg):
    "Convert canonical tool Msg to list of OpenAI Chat tool messages."
    return [denorm_openai_chat_tool_result(p) for p in m.content if p.type == PartType.tool_result]

def denorm_openai_chat_msgs(msgs:list[Msg]):
    "Convert list of canonical Msgs to OpenAI Chat messages."
    res = []
    for m in msgs:
        if   m.role == 'user':      res.append(denorm_openai_chat_user(m))
        elif m.role == 'assistant': res.extend(denorm_openai_chat_assistant(m))
        elif m.role == 'tool':      res.extend(denorm_openai_chat_tool(m))
    return res


#### Text:

In [ ]:
mn='gpt-4o-mini'
msg1 = Msg(role='user', content=[Part(type=PartType.text, text='Hi!')])
resp = await oai_cli.chat.create_chat_completion(model=mn,messages=denorm_openai_chat_msgs([msg1]),stream=True,stream_options={"include_usage": True})
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai_chat): pass
print(resp_comp.message.content[0].text)

Hello! How can I assist you today?


In [ ]:
denorm_openai_chat_msgs([msg1])

[{'role': 'user', 'content': 'Hi!'}]

In [ ]:
mn='gpt-4o-mini'
msg2,msg3 = resp_comp.message, Msg(role='user', content=[Part(type=PartType.text, text='How are you?')])
resp = await oai_cli.chat.create_chat_completion(model=mn, messages=denorm_openai_chat_msgs([msg1,msg2,msg3]),stream=True,stream_options={"include_usage": True})
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai_chat): pass
print(resp_comp.message.content[0].text)

I'm just a program, so I don't have feelings, but I'm here and ready to help you! How about you? How are you doing?


In [ ]:
denorm_openai_chat_msgs([msg1,msg2,msg3])

[{'role': 'user', 'content': 'Hi!'},
 {'role': 'assistant', 'content': 'Hello! How can I assist you today?'},
 {'role': 'user', 'content': 'How are you?'}]

#### Thinking:

In [ ]:
mn='kimi-k2.5'
msg1 = Msg(role='user', content=[Part(type=PartType.text, text='What is 31231231 * 12312?')])
resp = await kimi_cli.chat.create_chat_completion(model=mn,messages=denorm_openai_chat_msgs([msg1]),stream=True,stream_options={"include_usage": True})
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai_chat): pass
print(resp_comp.message.content)

[Part(type=<PartType.thinking: 'thinking'>, text='The user is asking for the product of 31,231,231 and 12,312.\n\nLet me calculate this step by step.\n\nFirst, I can break this down using the distributive property:\n31,231,231 × 12,312 = 31,231,231 × (12,000 + 300 + 12)\n\nOr I can do it directly:\n31,231,231 × 12,312\n\nLet me calculate:\n31,231,231 × 12,312 = ?\n\nBreaking it down:\n31,231,231 × 12,000 = 31,231,231 × 12 × 1,000 = 374,774,772 × 1,000 = 374,774,772,000\n\n31,231,231 × 300 = 31,231,231 × 3 × 100 = 93,693,693 × 100 = 9,369,369,300\n\n31,231,231 × 12 = 31,231,231 × 10 + 31,231,231 × 2 = 312,312,310 + 62,462,462 = 374,774,772\n\nNow add them up:\n374,774,772,000\n+   9,369,369,300\n+     374,774,772\n-------------------\n374,774,772,000\n  9,369,369,300\n    374,774,772\n---------------\n384,518,916,072\n\nLet me verify:\n374,774,772,000 + 9,369,369,300 = 384,144,141,300\n384,144,141,300 + 374,774,772 = 384,518,916,072\n\nSo the answer should be 384,518,916,072.\n\nLet me 

In [ ]:
denorm_openai_chat_msgs([msg1])

[{'role': 'user', 'content': 'What is 31231231 * 12312?'}]

In [ ]:
msg2,msg3 = resp_comp.message, Msg(role='user', content=[Part(type=PartType.text, text='Great!')])
resp = await kimi_cli.chat.create_chat_completion(model=mn, messages=denorm_openai_chat_msgs([msg1,msg2,msg3]),stream=True,stream_options={"include_usage": True})
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai_chat): pass
print(resp_comp.message.content[0].text)

The user said "Great!" which is a simple positive acknowledgment of my previous answer. They might be satisfied, or they might be setting up for a follow-up question. I should:

1. Acknowledge their response politely
2. Keep the door open for further questions
3. Not over-explain or ramble since they just gave a brief positive response

Since they seemed happy with the calculation, I can ask if they need help with anything else or if they had a specific use case for that multiplication. But keeping it brief is probably best.

I should avoid:
- Re-explaining the math (they already said "Great!")
- Asking why they needed that specific calculation (might be intrusive)
- Being overly formal or robotic

A simple, friendly response acknowledging their satisfaction and offering further help if needed would be appropriate.


In [ ]:
denorm_openai_chat_msgs([msg1,msg2,msg3])

[{'role': 'user', 'content': 'What is 31231231 * 12312?'},
 {'role': 'assistant',
  'content': "**384,518,916,072**\n\nHere's the calculation breakdown:\n\n31,231,231 × 12,312 can be computed by breaking 12,312 into 10,000 + 2,000 + 300 + 10 + 2:\n\n- 31,231,231 × 10,000 = 312,312,310,000\n- 31,231,231 × 2,000 = 62,462,462,000\n- 31,231,231 × 300 = 9,369,369,300\n- 31,231,231 × 10 = 312,312,310\n- 31,231,231 × 2 = 62,462,462\n\nAdding these together:\n312,312,310,000 + 62,462,462,000 = 374,774,772,000\n374,774,772,000 + 9,369,369,300 = 384,144,141,300\n384,144,141,300 + 312,312,310 = 384,456,453,610\n384,456,453,610 + 62,462,462 = **384,518,916,072**",
  'reasoning_content': 'The user is asking for the product of 31,231,231 and 12,312.\n\nLet me calculate this step by step.\n\nFirst, I can break this down using the distributive property:\n31,231,231 × 12,312 = 31,231,231 × (12,000 + 300 + 12)\n\nOr I can do it directly:\n31,231,231 × 12,312\n\nLet me calculate:\n31,231,231 × 12,312 = ?

#### Tool Call:

In [ ]:
mn='gpt-4o-mini'
msg1 = Msg(role='user', content=[Part(type=PartType.text, text='What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.')])
resp = await oai_cli.chat.create_chat_completion(model=mn,messages=denorm_openai_chat_msgs([msg1]), tools=[sch],stream=True,stream_options={"include_usage": True})
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai_chat): pass
print(resp_comp.message.content)

[Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'call_lH6XatachWXyyOIOHeSMr6Mh', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'server': False, 'index': 0, 'type': 'function'}), Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'call_ZMbiQm8Hah7xrNCNnUhUP9w5', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 20}, 'server': False, 'index': 1, 'type': 'function'})]


In [ ]:
denorm_openai_chat_msgs([msg1])

[{'role': 'user',
  'content': 'What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.'}]

In [ ]:
mn='gpt-4o-mini'
msg2,msg3 = resp_comp.message,mk_tool_res_msg(resp_comp.tool_use_parts, ('8', '30'))
resp = await oai_cli.chat.create_chat_completion(model=mn, messages=denorm_openai_chat_msgs([msg1,msg2,msg3]),stream=True,stream_options={"include_usage": True})
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.openai_chat): pass
print(resp_comp.message.content[0].text)

The results are as follows: 

- \(3 + 5 = 8\)
- \(10 + 20 = 30\)


In [ ]:
denorm_openai_chat_msgs([msg1,msg2,msg3])

[{'role': 'user',
  'content': 'What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.'},
 {'role': 'assistant',
  'tool_calls': [{'id': 'call_lH6XatachWXyyOIOHeSMr6Mh',
    'type': 'function',
    'function': {'name': 'simple_add', 'arguments': '{"a": 3, "b": 5}'}},
   {'id': 'call_ZMbiQm8Hah7xrNCNnUhUP9w5',
    'type': 'function',
    'function': {'name': 'simple_add', 'arguments': '{"a": 10, "b": 20}'}}]},
 {'role': 'tool',
  'tool_call_id': 'call_lH6XatachWXyyOIOHeSMr6Mh',
  'content': '8'},
 {'role': 'tool',
  'tool_call_id': 'call_ZMbiQm8Hah7xrNCNnUhUP9w5',
  'content': '30'}]

### Anthropic Messages Denorm

Ok let's write the anthropic denorm functions similarly. Have look at the previous functions, schema, and previous modules like streaming for reference to come up with them.

Again think carefully about across model provider support, we shouldn't pass raw data. If signature is looked up by Anthopric similar to openai reasoning id, then we should convert those into text like we do with openai. `server_tool_result` approach is correct it's a valid PartType and will be stored.

In [ ]:
#| export
def _ant_cc(block, p):
    "Add cache_control to block if present in Part.data."
    if (cc := (p.data or {}).get('cache_control')): block['cache_control'] = cc
    return block

def denorm_anthropic_tool_use(p:Part):
    "Convert canonical tool_use Part to Anthropic tool_use content block."
    d = p.data or {}
    block = dict(type='tool_use', id=d.get('id',''), name=d.get('name',''), input=d.get('arguments') or {})
    if 'caller' in d: block['caller'] = d['caller']
    return _ant_cc(block, p)

def denorm_anthropic_tool_result(p:Part):
    "Convert canonical tool_result Part to Anthropic tool_result content block."
    d = p.data or {}
    return _ant_cc(dict(type='tool_result', tool_use_id=d.get('id') or d.get('call_id',''), content=str(p.text)), p)

def denorm_anthropic_user(m:Msg):
    "Convert canonical user Msg to Anthropic user message."
    parts = [_ant_cc(dict(type='text', text=p.text or ''), p) for p in m.content if p.type == PartType.text]
    return dict(role='user', content=parts)

def denorm_anthropic_assistant(m:Msg):
    "Convert canonical assistant Msg to Anthropic assistant message + synthetic tool results for non-Anthropic server tools."
    blocks, srv_results, srv_call_id = [], [], None
    for p in m.content:
        if p.type == PartType.thinking:
            if srv_call_id:
                srv_results.append(dict(type='tool_result', tool_use_id=srv_call_id, content=p.text or ''))
                srv_call_id = None
            elif sig:=(p.data or {}).get('signature',''): blocks.append(dict(type='thinking', thinking=p.text or '', signature=sig))
            else:                               blocks.append(_ant_cc(dict(type='text', text=p.text or ''), p))
        elif p.type == PartType.text:
            if srv_call_id:
                srv_results.append(dict(type='tool_result', tool_use_id=srv_call_id, content=p.text or ''))
                srv_call_id = None
            else: blocks.append(_ant_cc(dict(type='text', text=p.text or ''), p))
        elif p.type == PartType.tool_use:
            if p.data.get('server') and (p.data.get('id','') or '').startswith('srvtoolu_'):
                blocks.append(dict(type='server_tool_use', id=p.data['id'], name=p.data.get('name',''), input=p.data.get('arguments') or {}))
            elif p.data.get('server'):
                blocks.append(denorm_anthropic_tool_use(p))
                srv_call_id = p.data.get('id','')
            else: blocks.append(denorm_anthropic_tool_use(p))
        elif p.type == PartType.server_tool_result: blocks.append(p.data if p.data else dict(type='server_tool_result'))
    res = [dict(role='assistant', content=blocks)]
    if srv_results: res.append(dict(role='user', content=srv_results))
    return res

def denorm_anthropic_tool(m:Msg):
    "Convert canonical tool Msg to Anthropic user message with tool_result blocks."
    blocks = [denorm_anthropic_tool_result(p) for p in m.content if p.type == PartType.tool_result]
    return [dict(role='user', content=blocks)]

def denorm_anthropic_msgs(msgs:list[Msg]):
    "Convert list of canonical Msgs to Anthropic messages."
    res = []
    for m in msgs:
        if   m.role == 'user':      res.append(denorm_anthropic_user(m))
        elif m.role == 'assistant': res.extend(denorm_anthropic_assistant(m))
        elif m.role == 'tool':      res.extend(denorm_anthropic_tool(m))
    return res

#### Text:

In [ ]:
mn,msg1 = 'claude-sonnet-4-20250514',Msg(role='user', content=[Part(type=PartType.text, text='Say hi')])
think,hdrs = {"type": "enabled", "budget_tokens": 5000}, {"anthropic-version": "2023-06-01"}
resp = await ant_cli.messages.messages_post(model=mn,messages=denorm_anthropic_msgs([msg1]),max_tokens=8000,thinking=think, headers=hdrs,stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.anthropic): pass
print(resp_comp.message.content)

[Part(type=<PartType.thinking: 'thinking'>, text='The human is asking me to say hi, which is a simple greeting. This is a straightforward request that I can fulfill easily.', data=None), Part(type=<PartType.text: 'text'>, text='Hi! How are you doing today?', data=None)]


In [ ]:
denorm_anthropic_msgs([msg1])

[{'role': 'user', 'content': [{'type': 'text', 'text': 'Say hi'}]}]

In [ ]:
msg2,msg3 = resp_comp.message, Msg(role='user', content=[Part(type=PartType.text, text='Great!')])
resp = await ant_cli.messages.messages_post(model=mn,messages=denorm_anthropic_msgs([msg1,msg2,msg3]),max_tokens=8000,thinking=think, headers=hdrs,stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.anthropic): pass
print(resp_comp.message.content)

[Part(type=<PartType.thinking: 'thinking'>, text='The human responded positively to my greeting, saying "Great!" This is a friendly, upbeat response. I should respond in kind with enthusiasm and perhaps ask a follow-up question to continue the conversation in a natural way.', data=None), Part(type=<PartType.text: 'text'>, text="That's wonderful to hear! Is there anything particular that's making your day great, or anything I can help you with?", data=None)]


In [ ]:
denorm_anthropic_msgs([msg1,msg2,msg3])

[{'role': 'user', 'content': [{'type': 'text', 'text': 'Say hi'}]},
 {'role': 'assistant',
  'content': [{'type': 'text',
    'text': 'The human is asking me to say hi, which is a simple greeting. This is a straightforward request that I can fulfill easily.'},
   {'type': 'text', 'text': 'Hi! How are you doing today?'}]},
 {'role': 'user', 'content': [{'type': 'text', 'text': 'Great!'}]}]

#### Tool Call

In [ ]:
mn,msg1 = 'claude-sonnet-4-20250514', Msg(role='user', content=[Part(type=PartType.text, text='What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.')])
think,hdrs = {"type": "enabled", "budget_tokens": 5000}, {"anthropic-version": "2023-06-01"}
tools = [{"name": "simple_add", "description": "add numbers", "input_schema": sch['function']['parameters']}]
resp = await ant_cli.messages.messages_post(model=mn, messages=denorm_anthropic_msgs([msg1]), max_tokens=8000, tools=tools, headers=hdrs, stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.anthropic): pass
print(resp_comp.message.content)

[Part(type=<PartType.text: 'text'>, text="I'll calculate both additions for you using the simple_add tool in parallel.", data=None), Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'toolu_01Tw4LmF2FmQvYohbMQoFAr2', 'name': 'simple_add', 'arguments': {'a': 3, 'b': 5}, 'server': False, 'caller': {'type': 'direct'}}), Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'toolu_0194h3QmEuiUpugJMBDZYfKw', 'name': 'simple_add', 'arguments': {'a': 10, 'b': 20}, 'server': False, 'caller': {'type': 'direct'}})]


In [ ]:
denorm_anthropic_msgs([msg1])

[{'role': 'user',
  'content': [{'type': 'text',
    'text': 'What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.'}]}]

In [ ]:
msg2,msg3 = resp_comp.message, mk_tool_res_msg(resp_comp.tool_use_parts, ('8', '30'))
resp = await ant_cli.messages.messages_post(model=mn,messages=denorm_anthropic_msgs([msg1,msg2,msg3]),max_tokens=8000,thinking=think, headers=hdrs,stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.anthropic): pass
print(resp_comp.message.content)

[Part(type=<PartType.text: 'text'>, text='The results are:\n- 3 + 5 = 8\n- 10 + 20 = 30', data=None)]


In [ ]:
denorm_anthropic_msgs([msg1,msg2,msg3])

[{'role': 'user',
  'content': [{'type': 'text',
    'text': 'What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.'}]},
 {'role': 'assistant',
  'content': [{'type': 'text',
    'text': "I'll calculate both additions for you using the simple_add tool in parallel."},
   {'type': 'tool_use',
    'id': 'toolu_01Tw4LmF2FmQvYohbMQoFAr2',
    'name': 'simple_add',
    'input': {'a': 3, 'b': 5},
    'caller': {'type': 'direct'}},
   {'type': 'tool_use',
    'id': 'toolu_0194h3QmEuiUpugJMBDZYfKw',
    'name': 'simple_add',
    'input': {'a': 10, 'b': 20},
    'caller': {'type': 'direct'}}]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01Tw4LmF2FmQvYohbMQoFAr2',
    'content': '8'},
   {'type': 'tool_result',
    'tool_use_id': 'toolu_0194h3QmEuiUpugJMBDZYfKw',
    'content': '30'}]}]

#### Web Search

In [ ]:
mn,msg1 = 'claude-sonnet-4-20250514', Msg(role='user', content=[Part(type=PartType.text, text='Use web search to get the weather in Istanbul?')])
think,hdrs = {"type": "enabled", "budget_tokens": 5000}, {"anthropic-version": "2023-06-01"}
tools=[{"type": "web_search_20250305", "name": "web_search"}]
resp = await ant_cli.messages.messages_post(model=mn, messages=denorm_anthropic_msgs([msg1]), tools=tools, max_tokens=8000, thinking=think, headers=hdrs, stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.anthropic): pass
L(resp_comp.message.content).attrgot('type')

[<PartType.thinking: 'thinking'>, <PartType.tool_use: 'tool_use'>, <PartType.server_tool_result: 'server_tool_result'>, <PartType.text: 'text'>]

In [ ]:
denorm_anthropic_msgs([msg1])

[{'role': 'user',
  'content': [{'type': 'text',
    'text': 'Use web search to get the weather in Istanbul?'}]}]

In [ ]:
msg2,msg3 = resp_comp.message, Msg(role='user', content=[Part(type=PartType.text, text='Great!')])
resp = await ant_cli.messages.messages_post(model=mn,messages=denorm_anthropic_msgs([msg1,msg2,msg3]),max_tokens=8000,thinking=think, headers=hdrs,stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.anthropic): pass
print(resp_comp.message.content)

[Part(type=<PartType.thinking: 'thinking'>, text='The user is expressing satisfaction with the weather information I provided for Istanbul. This is a simple positive response, so I should acknowledge their satisfaction in a friendly and brief way.', data=None), Part(type=<PartType.text: 'text'>, text="You're welcome! I'm glad I could help you get up-to-date weather information for Istanbul. The mild spring weather with temperatures in the 50s-60s°F (10-16°C) sounds quite pleasant, though it's good to be aware of the air quality situation. Let me know if you need weather information for any other locations or have any other questions!", data=None)]


### Gemini Generate Denorm

Next up is Gemini, please write the denorm function similar as before

In [ ]:
#| export
def denorm_gemini_tool_use(p:Part):
    "Convert canonical tool_use Part to Gemini functionCall part."
    d = p.data or {}
    fc = dict(name=d.get('name',''), args=d.get('arguments') or {})
    if d.get('id'): fc['id'] = d['id']
    part = dict(functionCall=fc)
    part['thoughtSignature'] = d.get('thoughtSignature', 'skip_thought_signature_validator')
    return part

def denorm_gemini_tool_result(p:Part):
    "Convert canonical tool_result Part to Gemini functionResponse part."
    d = p.data or {}
    fr = dict(name=d.get('name',''), response={"content": str(p.text)})
    if d.get('id'): fr['id'] = d['id']
    return dict(functionResponse=fr)

def denorm_gemini_user(m:Msg):
    "Convert canonical user Msg to Gemini user content."
    parts = [dict(text=p.text or '') for p in m.content if p.type == PartType.text]
    return dict(role='user', parts=parts)

def denorm_gemini_assistant(m:Msg):
    "Convert canonical assistant Msg to Gemini model content."
    parts = []
    for p in m.content:
        if   p.type == PartType.thinking: parts.append(dict(text=p.text or '', thought=True))
        elif p.type == PartType.text:     parts.append(dict(text=p.text or ''))
        elif p.type == PartType.tool_use: parts.append(denorm_gemini_tool_use(p))
    return dict(role='model', parts=parts)

def denorm_gemini_tool(m:Msg):
    "Convert canonical tool Msg to Gemini user content with functionResponse parts."
    parts = [denorm_gemini_tool_result(p) for p in m.content if p.type == PartType.tool_result]
    return dict(role='user', parts=parts)

def denorm_gemini_msgs(msgs:list[Msg]):
    "Convert list of canonical Msgs to Gemini contents."
    res = []
    for m in msgs:
        if   m.role == 'user':      res.append(denorm_gemini_user(m))
        elif m.role == 'assistant': res.append(denorm_gemini_assistant(m))
        elif m.role == 'tool':      res.append(denorm_gemini_tool(m))
    return res

#### Text:

Demonstrating how canonical back and forth provider translations work

In [ ]:
mn,msg1 = 'models/gemini-3-flash-preview',Msg(role='user', content=[Part(type=PartType.text, text='Hi how are you?')])
resp = await gem_cli.models.stream_generate_content(model=mn, contents=denorm_gemini_msgs([msg1]), stream=True, _query={"alt": "sse"})
async for comp in acollect_stream(resp, model=mn, api_name=ApiName.gemini): pass
print(comp.message.content[0].text)

I'm doing great, thank you for asking! How are you doing today? Is there anything I can help you with?


In [ ]:
denorm_gemini_msgs([msg1])

[{'role': 'user', 'parts': [{'text': 'Hi how are you?'}]}]

In [ ]:
msg2,msg3 = comp.message, Msg(role='user', content=[Part(type=PartType.text, text='Great!')])
resp = await gem_cli.models.stream_generate_content(model=mn, contents=denorm_gemini_msgs([msg1,msg2,msg3]), stream=True, _query={"alt": "sse"})
async for comp in acollect_stream(resp, model=mn, api_name=ApiName.gemini): pass
print(comp.message.content[0].text)

That's wonderful to hear! 

Is there anything specific on your mind today, or anything I can help you with? I'm ready for whatever you need—whether it's answering a question, helping with a project, or just chatting!


In [ ]:
denorm_gemini_msgs([msg1,msg2,msg3])

[{'role': 'user', 'parts': [{'text': 'Hi how are you?'}]},
 {'role': 'model',
  'parts': [{'text': "I'm doing great, thank you for asking! How are you doing today? Is there anything I can help you with?"}]},
 {'role': 'user', 'parts': [{'text': 'Great!'}]}]

In [ ]:
denorm_gemini_msgs([msg1,msg2,msg3])

[{'role': 'user', 'parts': [{'text': 'Hi how are you?'}]},
 {'role': 'model',
  'parts': [{'text': "I'm doing great, thank you for asking! How are you doing today? Is there anything I can help you with?"}]},
 {'role': 'user', 'parts': [{'text': 'Great!'}]}]

#### Thinking

In [ ]:
mn,msg1 = 'models/gemini-3-flash-preview',Msg(role='user', content=[Part(type=PartType.text, text='What is 31231231 * 12312?')])
resp = await gem_cli.models.stream_generate_content(model=mn, contents=denorm_gemini_msgs([msg1]), generation_config={"thinking_config": {"include_thoughts": True}}, stream=True, _query={"alt": "sse"})
async for comp in acollect_stream(resp, model=mn, api_name=ApiName.gemini): pass
comp.message.content

[Part(type=<PartType.thinking: 'thinking'>, text="**Considering Multiplication Options**\n\nI'm currently evaluating the best approach for multiplying these large numbers. I've considered long multiplication, which is the most straightforward, though potentially tedious. Decomposition, using the distributive property, is the other method I'm currently thinking of, potentially breaking down one of the numbers for simplification. I am looking for the method that will minimize calculation errors.\n\n\n**Calculating Product Breakdown**\n\nI've now performed the multiplication using the distributive property, breaking down the factors for easier computation. I broke down 12,312 into components and then calculated each product, verifying with a standard method. My answer seems correct, though this method is time consuming, so I'll consider alternative methods for larger numbers.\n\n\n**Verifying Partial Sums**\n\nI've been meticulously checking the partial products and their summation. I'm f

In [ ]:
denorm_gemini_msgs([msg1])

[{'role': 'user', 'parts': [{'text': 'What is 31231231 * 12312?'}]}]

In [ ]:
msg2,msg3 = comp.message, Msg(role='user', content=[Part(type=PartType.text, text='Great!')])
resp = await gem_cli.models.stream_generate_content(model=mn, contents=denorm_gemini_msgs([msg1,msg2,msg3]), generation_config={"thinking_config": {"include_thoughts": True}}, stream=True, _query={"alt": "sse"})
async for comp in acollect_stream(resp, model=mn, api_name=ApiName.gemini): pass
comp.message.content

[Part(type=<PartType.text: 'text'>, text="You're welcome! If you have any other calculations or questions, feel free to ask. I'm happy to help!", data=None)]

In [ ]:
denorm_gemini_msgs([msg1,msg2,msg3])

[{'role': 'user', 'parts': [{'text': 'What is 31231231 * 12312?'}]},
 {'role': 'model',
  'parts': [{'text': "**Considering Multiplication Options**\n\nI'm currently evaluating the best approach for multiplying these large numbers. I've considered long multiplication, which is the most straightforward, though potentially tedious. Decomposition, using the distributive property, is the other method I'm currently thinking of, potentially breaking down one of the numbers for simplification. I am looking for the method that will minimize calculation errors.\n\n\n**Calculating Product Breakdown**\n\nI've now performed the multiplication using the distributive property, breaking down the factors for easier computation. I broke down 12,312 into components and then calculated each product, verifying with a standard method. My answer seems correct, though this method is time consuming, so I'll consider alternative methods for larger numbers.\n\n\n**Verifying Partial Sums**\n\nI've been meticulou

#### Tool Call

In [ ]:
msg1 = Msg(role='user', content=[Part(type=PartType.text, text=inp)])
resp = await gem_cli.models.stream_generate_content(model=mn, contents=denorm_gemini_msgs([msg1]), tools=[{"functionDeclarations": [sch['function']]}], stream=True, _query={"alt": "sse"})
async for comp in acollect_stream(resp, model=mn, api_name=ApiName.gemini): pass
comp.message.content

[Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': 'vchmcu53', 'name': 'simple_add', 'arguments': {'b': 5, 'a': 3}, 'server': False, 'thoughtSignature': 'EqUDCqIDAb4+9vvgiBWJe8rHQ4AsefEidQLNf3wESYqUssWiAripZiGOGQlKRycajyxuPB7kSU1sCX6zg/ULNAjMY9qklM9iUN38J2LqKwgUjJ6JC1grDOj+VxBT4DuB+2NZ1BKk5+pxMSp9/1vkArpfMN8ekYx8nk4izcpODjj846vfl8sKFu0sMHjDazItyTUrfPFniUVcX8aGbdMVjQQt5jh9DcFI5nBh8LD1hrlck29IRvwnL0OcbyndUDyKewMS8msM22dsYE4gJg0AqrLcCsKBb+arW9zLWQFcC85qEtCLs+EQMdk6pTSYQOrZf2KJZJHXW3kv11ioq8p81aSyq6xnAUSDs9og7azl5XKyL+nWXe5Zum3aCyjva4NEDpIotMvrW0wH1cXPaR7WPqxVg9qk/YNuGgRlZ2cQDrKLuge1/z9/vk9v4w4/t24sJzYJRdFH4RWgdTqmHXhFz78Q62hAQtHXHpWXnwHtrQ/yRyzdimy3DC2XHt9SbEDd4ULaJy/AI0EyyDJTFfZALh2F5tyLf7iK929aVChkAUUg6p3AsR95XQ=='}),
 Part(type=<PartType.tool_use: 'tool_use'>, text=None, data={'id': '7rwckpni', 'name': 'simple_add', 'arguments': {'b': 20, 'a': 10}, 'server': False})]

In [ ]:
denorm_openai_responses_msgs([msg1])

[{'type': 'message',
  'role': 'user',
  'content': [{'type': 'input_text',
    'text': 'What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.'}]}]

In [ ]:
tmsg = mk_tool_res_msg(comp.tool_use_parts, (8,30)); tmsg

Msg(role='tool', content=[Part(type=<PartType.tool_result: 'tool_result'>, text=8, data={'id': 'vchmcu53', 'name': 'simple_add', 'arguments': {'b': 5, 'a': 3}, 'server': False, 'thoughtSignature': 'EqUDCqIDAb4+9vvgiBWJe8rHQ4AsefEidQLNf3wESYqUssWiAripZiGOGQlKRycajyxuPB7kSU1sCX6zg/ULNAjMY9qklM9iUN38J2LqKwgUjJ6JC1grDOj+VxBT4DuB+2NZ1BKk5+pxMSp9/1vkArpfMN8ekYx8nk4izcpODjj846vfl8sKFu0sMHjDazItyTUrfPFniUVcX8aGbdMVjQQt5jh9DcFI5nBh8LD1hrlck29IRvwnL0OcbyndUDyKewMS8msM22dsYE4gJg0AqrLcCsKBb+arW9zLWQFcC85qEtCLs+EQMdk6pTSYQOrZf2KJZJHXW3kv11ioq8p81aSyq6xnAUSDs9og7azl5XKyL+nWXe5Zum3aCyjva4NEDpIotMvrW0wH1cXPaR7WPqxVg9qk/YNuGgRlZ2cQDrKLuge1/z9/vk9v4w4/t24sJzYJRdFH4RWgdTqmHXhFz78Q62hAQtHXHpWXnwHtrQ/yRyzdimy3DC2XHt9SbEDd4ULaJy/AI0EyyDJTFfZALh2F5tyLf7iK929aVChkAUUg6p3AsR95XQ=='}), Part(type=<PartType.tool_result: 'tool_result'>, text=30, data={'id': '7rwckpni', 'name': 'simple_add', 'arguments': {'b': 20, 'a': 10}, 'server': False})], data=None)

In [ ]:
msg2 = comp.message
resp = await gem_cli.models.stream_generate_content(model=mn, contents=denorm_gemini_msgs([msg1,msg2,tmsg]), tools=[{"functionDeclarations": [sch['function']]}], stream=True, _query={"alt": "sse"})
async for comp in acollect_stream(resp, model=mn, api_name=ApiName.gemini): pass
comp.message.content

[Part(type=<PartType.text: 'text'>, text='OK. 3 + 5 is 8, and 10 + 20 is 30.', data=None)]

In [ ]:
denorm_gemini_msgs([msg1,msg2,tmsg])

[{'role': 'user',
  'parts': [{'text': 'What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.'}]},
 {'role': 'model',
  'parts': [{'functionCall': {'name': 'simple_add',
     'args': {'b': 5, 'a': 3},
     'id': 'vchmcu53'},
    'thoughtSignature': 'EqUDCqIDAb4+9vvgiBWJe8rHQ4AsefEidQLNf3wESYqUssWiAripZiGOGQlKRycajyxuPB7kSU1sCX6zg/ULNAjMY9qklM9iUN38J2LqKwgUjJ6JC1grDOj+VxBT4DuB+2NZ1BKk5+pxMSp9/1vkArpfMN8ekYx8nk4izcpODjj846vfl8sKFu0sMHjDazItyTUrfPFniUVcX8aGbdMVjQQt5jh9DcFI5nBh8LD1hrlck29IRvwnL0OcbyndUDyKewMS8msM22dsYE4gJg0AqrLcCsKBb+arW9zLWQFcC85qEtCLs+EQMdk6pTSYQOrZf2KJZJHXW3kv11ioq8p81aSyq6xnAUSDs9og7azl5XKyL+nWXe5Zum3aCyjva4NEDpIotMvrW0wH1cXPaR7WPqxVg9qk/YNuGgRlZ2cQDrKLuge1/z9/vk9v4w4/t24sJzYJRdFH4RWgdTqmHXhFz78Q62hAQtHXHpWXnwHtrQ/yRyzdimy3DC2XHt9SbEDd4ULaJy/AI0EyyDJTFfZALh2F5tyLf7iK929aVChkAUUg6p3AsR95XQ=='},
   {'functionCall': {'name': 'simple_add',
     'args': {'b': 20, 'a': 10},
     'id': '7rwckpni'},
    'thoughtSignature': 'skip_thought_signature_validator'}]},
 {'ro

Dummy `thoughtSignature` test:

In [ ]:
mn,msg1 = 'claude-sonnet-4-20250514', Msg(role='user', content=[Part(type=PartType.text, text='What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.')])
think,hdrs = {"type": "enabled", "budget_tokens": 5000}, {"anthropic-version": "2023-06-01"}
tools = [{"name": "simple_add", "description": "add numbers", "input_schema": sch['function']['parameters']}]
resp = await ant_cli.messages.messages_post(model=mn, messages=denorm_anthropic_msgs([msg1]), max_tokens=8000, tools=tools, headers=hdrs, stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.anthropic): pass
msg2,msg3 = resp_comp.message, mk_tool_res_msg(resp_comp.tool_use_parts, ('8', '30'))
resp = await ant_cli.messages.messages_post(model=mn,messages=denorm_anthropic_msgs([msg1,msg2,msg3]),max_tokens=8000,thinking=think, headers=hdrs,stream=True)
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.anthropic): pass
msg4 = resp_comp.message

In [ ]:
msg5 = Msg(role='user', content=[Part(type=PartType.text, text='Nice work Gemini!')])
msgs = denorm_gemini_msgs([msg1,msg2,msg3,msg4,msg5]); msgs

[{'role': 'user',
  'parts': [{'text': 'What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.'}]},
 {'role': 'model',
  'parts': [{'text': "I'll calculate both additions for you using the simple_add tool in parallel."},
   {'functionCall': {'name': 'simple_add',
     'args': {'a': 3, 'b': 5},
     'id': 'toolu_01Tw4LmF2FmQvYohbMQoFAr2'},
    'thoughtSignature': 'skip_thought_signature_validator'},
   {'functionCall': {'name': 'simple_add',
     'args': {'a': 10, 'b': 20},
     'id': 'toolu_0194h3QmEuiUpugJMBDZYfKw'},
    'thoughtSignature': 'skip_thought_signature_validator'}]},
 {'role': 'user',
  'parts': [{'functionResponse': {'name': 'simple_add',
     'response': {'content': '8'},
     'id': 'toolu_01Tw4LmF2FmQvYohbMQoFAr2'}},
   {'functionResponse': {'name': 'simple_add',
     'response': {'content': '30'},
     'id': 'toolu_0194h3QmEuiUpugJMBDZYfKw'}}]},
 {'role': 'model',
  'parts': [{'text': 'The results are:\n- 3 + 5 = 8\n- 10 + 20 = 30'}]},
 {'role': 'user', 'parts

In [ ]:
mn = 'models/gemini-3-flash-preview'
resp = await gem_cli.models.stream_generate_content(model=mn, contents=msgs, tools=[{"functionDeclarations": [sch['function']]}], stream=True, _query={"alt": "sse"})
async for comp in acollect_stream(resp, model=mn, api_name=ApiName.gemini): pass
comp.message.content

[Part(type=<PartType.text: 'text'>, text="You're very welcome! I'm glad I could help. Let me know if you have any more additions or other questions!", data=None)]

Are we creating a dummy `thoughtSignature` in case it's missing, e.g. switching over to gemini from another provider? I wasn't expecting this to work!

Please check the detailed notes about `thoughtSignature` from earlier dialogs

In [ ]:
# disable_cachy()

In [ ]:
mn = 'models/gemini-3-flash-preview'
resp = await gem_cli.models.stream_generate_content(model=mn, contents=msgs, tools=[{"functionDeclarations": [sch['function']]}], stream=True, _query={"alt": "sse"})
async for comp in acollect_stream(resp, model=mn, api_name=ApiName.gemini): pass
comp.message.content

[Part(type=<PartType.text: 'text'>, text="You're very welcome! I'm glad I could help. Let me know if you have any more additions or other questions!", data=None)]

Works without cachy too

In [ ]:
# disable_cachy()

In [ ]:
mn = 'models/gemini-3-flash-preview'
resp = await gem_cli.models.stream_generate_content(model=mn, contents=msgs, tools=[{"functionDeclarations": [sch['function']]}], generation_config={"thinking_config": {"include_thoughts": True}}, stream=True, _query={"alt": "sse"})
async for comp in acollect_stream(resp, model=mn, api_name=ApiName.gemini): pass
comp.message.content

[Part(type=<PartType.text: 'text'>, text="You're very welcome! I'm glad I could help. Let me know if you have any other questions!", data=None)]

Even with thinking it works

#### Web Search

In [ ]:
msg1 = Msg(role='user', content=[Part(type=PartType.text, text='What is the weather in Istanbul today?')])
resp = await gem_cli.models.stream_generate_content(model=mn, contents=denorm_gemini_msgs([msg1]), tools=[{"googleSearch": {}}], stream=True, _query={"alt": "sse"})
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.gemini): pass
print(resp_comp.message.content)

[Part(type=<PartType.text: 'text'>, text='Today in Istanbul, the weather is **partly sunny** with comfortable spring temperatures.\n\nHere is the detailed forecast for Monday, April 13, 2026:\n\n*   **Temperature:** The high today is approximately **14°C (58°F)**, with an overnight low reaching **6°C (43°F)**.\n*   **Conditions:** It will be mostly sunny throughout the afternoon, becoming partly cloudy in the evening.\n*   **Precipitation:** There is a very low chance of rain (around 3–10%).\n*   **Humidity:** The humidity level is moderate at about **45–53%**.\n*   **Wind:** Expect light breezes, typical for this time of year in the region.\n\nOverall, it is a pleasant day for outdoor activities, though you may want a light jacket for the cooler morning and evening hours.', data=None)]


In [ ]:
denorm_openai_responses_msgs([msg1])

[{'type': 'message',
  'role': 'user',
  'content': [{'type': 'input_text',
    'text': 'What is the weather in Istanbul today?'}]}]

In [ ]:
msg2,msg3 = resp_comp.message, Msg(role='user', content=[Part(type=PartType.text, text='Great!')])
resp = await gem_cli.models.stream_generate_content(model=mn, contents=denorm_gemini_msgs([msg1,msg2,msg3]), tools=[{"googleSearch": {}}], stream=True, _query={"alt": "sse"})
async for resp_comp in acollect_stream(resp, model=mn, api_name=ApiName.gemini): pass
print(resp_comp.message.content)

[Part(type=<PartType.text: 'text'>, text="You're welcome! Since today is actually **Wednesday, April 15, 2026**, the weather has warmed up a bit more compared to the start of the week.\n\nHere is the update for Istanbul today:\n\n*   **Temperature:** A high of **17°C (63°F)** and a low of **9°C (48°F)**.\n*   **Conditions:** Mostly clear skies with plenty of sunshine throughout the day.\n*   **Wind:** A gentle breeze from the North-Northeast at about 15 km/h.\n*   **Forecast for Tomorrow:** It looks like the sunny spell will continue into Thursday with temperatures climbing slightly higher.\n\nIt’s a perfect day to be out by the Bosphorus! Is there anything else you’d like to know?", data=None)]


In [ ]:
denorm_openai_responses_msgs([msg1,msg2,msg3])

[{'type': 'message',
  'role': 'user',
  'content': [{'type': 'input_text',
    'text': 'What is the weather in Istanbul today?'}]},
 {'type': 'message',
  'role': 'assistant',
  'content': [{'type': 'output_text',
    'text': 'Today in Istanbul, the weather is **partly sunny** with comfortable spring temperatures.\n\nHere is the detailed forecast for Monday, April 13, 2026:\n\n*   **Temperature:** The high today is approximately **14°C (58°F)**, with an overnight low reaching **6°C (43°F)**.\n*   **Conditions:** It will be mostly sunny throughout the afternoon, becoming partly cloudy in the evening.\n*   **Precipitation:** There is a very low chance of rain (around 3–10%).\n*   **Humidity:** The humidity level is moderate at about **45–53%**.\n*   **Wind:** Expect light breezes, typical for this time of year in the region.\n\nOverall, it is a pleasant day for outdoor activities, though you may want a light jacket for the cooler morning and evening hours.'}]},
 {'type': 'message',
  'r

### acomplete

We've successfully created an initial version for cross provider msg conversion, now we can create a high level function to swap model between to test all to all provider support. After that we can work on remaining integrations such as anthropic caching, media inputs etc..

To start with we can look at canonical `RequestOptions` to decide the signature of this function and work from there

In [ ]:
RequestOptions??

Object `RequestOptions` not found.


For our test I think we first need to provide a canonicalized way (e.g. openai chat format) and have their provider specific denorm functions for `tools`, `tool_choice`, `max_tokens`, `temperature`, `reasoning_effort` wdyt?

Sure

Good call

#### Tool Schemas

In [ ]:
#| export
def _fn_schema(t):
    "Extract (name, description, parameters) from any tool format."
    if not isinstance(t, dict): return None
    fn = t.get('function')
    if isinstance(fn, dict): return fn.get('name',''), fn.get('description',''), fn.get('parameters',{})
    if 'name' in t and ('parameters' in t or 'input_schema' in t):
        return t.get('name',''), t.get('description',''), t.get('parameters', t.get('input_schema',{}))
    return None

def denorm_openai_responses_tools(tools):
    "Convert canonical tools to OpenAI Responses format."
    out = []
    for t in tools:
        fn = _fn_schema(t)
        if fn is None: out.append(t); continue
        name, desc, params = fn
        out.append(dict(type='function', name=name, description=desc, parameters=params))
    return out

def denorm_openai_chat_tools(tools):
    "Passthrough — canonical format is already OpenAI Chat."
    return tools

def denorm_anthropic_tools(tools):
    "Convert canonical tools to Anthropic format."
    out = []
    for t in tools:
        fn = _fn_schema(t)
        if fn is None: out.append(t); continue
        name, desc, params = fn
        out.append(dict(name=name, description=desc, input_schema=params))
    return out

def denorm_gemini_tools(tools):
    "Convert canonical tools to Gemini format."
    fn_decls, other = [], []
    for t in tools:
        fn = _fn_schema(t)
        if fn is None: other.append(t); continue
        name, desc, params = fn
        fn_decls.append(dict(name=name, description=desc, parameters=params))
    out = other[:]
    if fn_decls: out.insert(0, dict(functionDeclarations=fn_decls))
    return out

Add tests for each provider with a user defined function schema, and non-function tools

In [ ]:
sch

{'type': 'function',
 'function': {'name': 'simple_add',
  'description': 'add numbers',
  'parameters': {'type': 'object',
   'properties': {'a': {'description': '', 'type': 'integer'},
    'b': {'description': '', 'type': 'integer'}},
   'required': ['a', 'b']}}}

In [ ]:
# Canonical tool (OpenAI Chat format) + non-function tools per provider
fn_tool = sch  # {"type": "function", "function": {"name": "simple_add", ...}}

# OpenAI Responses
test_eq(denorm_openai_responses_tools([fn_tool]), [{'type': 'function', 'name': 'simple_add', 'description': 'add numbers', 'parameters': sch['function']['parameters']}])
test_eq(denorm_openai_responses_tools([{"type": "web_search_preview"}]), [{"type": "web_search_preview"}])

# OpenAI Chat (passthrough)
test_eq(denorm_openai_chat_tools([fn_tool]), [fn_tool])

# Anthropic
test_eq(denorm_anthropic_tools([fn_tool]), [{'name': 'simple_add', 'description': 'add numbers', 'input_schema': sch['function']['parameters']}])
test_eq(denorm_anthropic_tools([{"type": "web_search_20250305", "name": "web_search"}]), [{"type": "web_search_20250305", "name": "web_search"}])

# Gemini
test_eq(denorm_gemini_tools([fn_tool]), [{'functionDeclarations': [{'name': 'simple_add', 'description': 'add numbers', 'parameters': sch['function']['parameters']}]}])
test_eq(denorm_gemini_tools([{"googleSearch": {}}]), [{"googleSearch": {}}])
test_eq(denorm_gemini_tools([fn_tool, {"googleSearch": {}}]), [{'functionDeclarations': [{'name': 'simple_add', 'description': 'add numbers', 'parameters': sch['function']['parameters']}]}, {"googleSearch": {}}])

#### Tool Choice

.

Yes, write helpers here and add tests

what about specifying a tool name in tool_choice?

Yes please, before that see if there are other things you are missing that we can support in canonical args?

> Dict passthrough: provider-native dicts pass through as-is

explain this please

Let's keep it simple. We will anyway think about how to expose the following next:

```
native: dict = None
extra_body: dict = None
extra_headers: dict = None
extra_query: dict = None
```

In [ ]:
#| export
_tc_modes = {'auto', 'required', 'any', 'force', 'none', 'off', 'disabled'}

def denorm_openai_responses_tool_choice(v):
    "Map canonical tool_choice to OpenAI Responses format."
    if v is None: return None
    s = str(v).strip().lower()
    if s in _tc_modes: return s if s in ('auto','none','required') else {'auto':'auto','any':'required','force':'required','off':'none','disabled':'none'}[s]
    return {'type': 'function', 'name': v}

def denorm_openai_chat_tool_choice(v):
    "Map canonical tool_choice to OpenAI Chat format."
    if v is None: return None
    s = str(v).strip().lower()
    if s in _tc_modes: return s if s in ('auto','none','required') else {'any':'required','force':'required','off':'none','disabled':'none'}[s]
    return {'type': 'function', 'function': {'name': v}}

def denorm_anthropic_tool_choice(v):
    "Map canonical tool_choice to Anthropic format."
    if v is None: return None
    s = str(v).strip().lower()
    if s in ('auto',):                    return {'type': 'auto'}
    if s in ('required', 'any', 'force'): return {'type': 'any'}
    if s in ('none', 'off', 'disabled'):  return None
    return {'type': 'tool', 'name': v}

def denorm_gemini_tool_choice(v):
    "Map canonical tool_choice to Gemini toolConfig."
    if v is None: return None
    s = str(v).strip().lower()
    if s in ('auto',):                    return {'functionCallingConfig': {'mode': 'AUTO'}}
    if s in ('required', 'any', 'force'): return {'functionCallingConfig': {'mode': 'ANY'}}
    if s in ('none', 'off', 'disabled'):  return {'functionCallingConfig': {'mode': 'NONE'}}
    return {'functionCallingConfig': {'mode': 'ANY', 'allowedFunctionNames': [v]}}

In [ ]:
# Modes
test_eq(denorm_openai_responses_tool_choice('auto'), 'auto')
test_eq(denorm_openai_responses_tool_choice('required'), 'required')
test_eq(denorm_openai_chat_tool_choice('required'), 'required')
test_eq(denorm_anthropic_tool_choice('auto'), {'type': 'auto'})
test_eq(denorm_anthropic_tool_choice('required'), {'type': 'any'})
test_eq(denorm_anthropic_tool_choice('none'), None)
test_eq(denorm_gemini_tool_choice('auto'), {'functionCallingConfig': {'mode': 'AUTO'}})
test_eq(denorm_gemini_tool_choice('none'), {'functionCallingConfig': {'mode': 'NONE'}})

# Tool name
test_eq(denorm_openai_responses_tool_choice('simple_add'), {'type': 'function', 'name': 'simple_add'})
test_eq(denorm_openai_chat_tool_choice('simple_add'), {'type': 'function', 'function': {'name': 'simple_add'}})
test_eq(denorm_anthropic_tool_choice('simple_add'), {'type': 'tool', 'name': 'simple_add'})
test_eq(denorm_gemini_tool_choice('simple_add'), {'functionCallingConfig': {'mode': 'ANY', 'allowedFunctionNames': ['simple_add']}})

# None
test_eq(denorm_openai_responses_tool_choice(None), None)
test_eq(denorm_anthropic_tool_choice(None), None)
test_eq(denorm_gemini_tool_choice(None), None)

#### Reasoning Effort

Ok let's do other args now

In [ ]:
#| export
def denorm_openai_responses_reasoning(v):
    "Map canonical reasoning_effort to OpenAI Responses reasoning param."
    if v is None: return None
    return {'effort': v} if isinstance(v, str) else v

def denorm_openai_chat_reasoning(v): return v  # passthrough as reasoning_effort param

_ant_think_budgets = dict(minimal=1024, low=2048, medium=4096, high=8192, max=16384)
def denorm_anthropic_reasoning(v):
    "Map canonical reasoning_effort to Anthropic thinking config."
    if v is None: return None
    if isinstance(v, dict): return v
    return {'type': 'enabled', 'budget_tokens': _ant_think_budgets.get(str(v).lower(), 4096)}

_gem_think_budgets = dict(minimal=128, low=1024, medium=2048, high=4096)
_gem_think_levels  = dict(minimal='low', low='low', medium='medium', high='high')

def denorm_gemini_reasoning(v, model=''):
    "Map canonical reasoning_effort to Gemini thinkingConfig (uses thinkingLevel for Gemini 3+)."
    if v is None: return None
    if isinstance(v, dict): return v
    k = str(v).lower()
    # defaults to includeThoughts same as litellm
    if 'gemini-3' in model: return {'thinkingLevel': _gem_think_levels.get(k, 'medium'), 'includeThoughts': True}
    return {'thinkingBudget': _gem_think_budgets.get(str(v).lower(), 1024), 'includeThoughts': True}

In [ ]:
# reasoning_effort
test_eq(denorm_openai_responses_reasoning('low'), {'effort': 'low'})
test_eq(denorm_openai_responses_reasoning(None), None)
test_eq(denorm_openai_chat_reasoning('low'), 'low')
test_eq(denorm_anthropic_reasoning('low'), {'type': 'enabled', 'budget_tokens': 2048})
test_eq(denorm_anthropic_reasoning('high'), {'type': 'enabled', 'budget_tokens': 8192})
test_eq(denorm_anthropic_reasoning(None), None)
test_eq(denorm_gemini_reasoning('medium', 'models/gemini-2.5-flash'), {'thinkingBudget': 2048, 'includeThoughts': True})
test_eq(denorm_gemini_reasoning('low', 'models/gemini-3-flash-preview'), {'thinkingLevel': 'low', 'includeThoughts': True})
test_eq(denorm_gemini_reasoning('high', 'models/gemini-3-flash-preview'), {'thinkingLevel': 'high', 'includeThoughts': True})
test_eq(denorm_gemini_reasoning(None), None)

.

if stream it should yield all responses that we collect with `acollect`

It should use the non-stream path. You can see example non-stream calls in previous dialogs. As for cli detection let's build them on the fly for now and make provider ane explicit arg to the function. Later we can infer it 

Yes it should return directly for non streaming

How does litellm handle it in `acomplete`?

Ok so we just return `acollect_stream(resp, model=model, provider=p)`?

Sounds good

#### acomplete

In [ ]:
def _sys_text(system):
    "Extract text from system (str or Part)."
    if system is None: return None
    return system if isinstance(system, str) else system.text

async def acomplete(msgs, model, api_name, stream=False, system=None, max_tokens=None, temperature=None, tools=None, tool_choice=None, reasoning_effort=None, **kwargs):
    "Unified completion across providers."
    if api_name == ApiName.openai:
        cli = OpenAPIClient(oai_spec, headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"})
        payload = dict(model=model, input=denorm_openai_responses_msgs(msgs), stream=stream)
        if system:           payload['instructions'] = _sys_text(system)
        if max_tokens:       payload['max_output_tokens'] = max_tokens
        if temperature is not None: payload['temperature'] = temperature
        if tools:            payload['tools'] = denorm_openai_responses_tools(tools)
        if tool_choice:      payload['tool_choice'] = denorm_openai_responses_tool_choice(tool_choice)
        if reasoning_effort: payload['reasoning'] = denorm_openai_responses_reasoning(reasoning_effort)
        resp = await cli.responses.create_response(**payload, **kwargs)
        if stream: return acollect_stream(resp, model=model, api_name=api_name)
        return normalize_openai_response(resp, model=model, api_name=api_name)

    elif api_name == ApiName.openai_chat:
        if 'kimi' in model.lower():
            cli = OpenAPIClient(oai_spec, headers={"Authorization": f"Bearer {os.environ['MOONSHOT_API_KEY']}"})
            for op in cli.ops: op.base_url = 'https://api.moonshot.ai/v1'
        else:
            cli = OpenAPIClient(oai_spec, headers={"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"})
        payload = dict(model=model, messages=denorm_openai_chat_msgs(msgs), stream=stream)
        if system:           payload['messages'].insert(0, {'role': 'system', 'content': _sys_text(system)})
        if stream: payload['stream_options'] = {"include_usage": True}
        if max_tokens:       payload['max_tokens'] = max_tokens
        if temperature is not None: payload['temperature'] = temperature
        if tools:            payload['tools'] = denorm_openai_chat_tools(tools)
        if tool_choice:      payload['tool_choice'] = denorm_openai_chat_tool_choice(tool_choice)
        if reasoning_effort: payload['reasoning_effort'] = denorm_openai_chat_reasoning(reasoning_effort)
        resp = await cli.chat.create_chat_completion(**payload, **kwargs)
        if stream: return acollect_stream(resp, model=model, api_name=api_name)
        return normalize_openai_chat_completion(resp, model=model, api_name=api_name)

    elif api_name == ApiName.anthropic:
        cli = OpenAPIClient(ant_spec, headers={"x-api-key": os.environ["ANTHROPIC_API_KEY"], "anthropic-version": "2023-06-01"})
        payload = dict(model=model, messages=denorm_anthropic_msgs(msgs), max_tokens=max_tokens or 1024, stream=stream)
        if system:
            if isinstance(system, Part):
                block = {'type': 'text', 'text': system.text or ''}
                if (cc := (system.data or {}).get('cache_control')): block['cache_control'] = cc
                payload['system'] = [block]
            else: payload['system'] = system
        if temperature is not None: payload['temperature'] = temperature
        if tools:            payload['tools'] = denorm_anthropic_tools(tools)
        tc = denorm_anthropic_tool_choice(tool_choice)
        if tc:               payload['tool_choice'] = tc
        think = denorm_anthropic_reasoning(reasoning_effort)
        if think:
            payload['thinking'] = think
            bt = think.get('budget_tokens', 0)
            if payload['max_tokens'] <= bt: payload['max_tokens'] = bt + 256
        resp = await cli.messages.messages_post(**payload, **kwargs)
        if stream: return acollect_stream(resp, model=model, api_name=api_name)
        return normalize_anthropic_message(resp, model=model, api_name=api_name)

    elif api_name == ApiName.gemini:
        cli = OpenAPIClient(gem_spec, headers={"x-goog-api-key": os.environ["GEMINI_API_KEY"]})
        gen_config = {}
        if max_tokens:       gen_config['maxOutputTokens'] = max_tokens
        if temperature is not None: gen_config['temperature'] = temperature
        think = denorm_gemini_reasoning(reasoning_effort,model)
        if think:            gen_config['thinkingConfig'] = think
        payload = dict(model=model, contents=denorm_gemini_msgs(msgs))
        if system:           payload['system_instruction'] = {'parts': [{'text': _sys_text(system)}]}
        if gen_config:       payload['generation_config'] = gen_config
        if tools:
            gem_tools = denorm_gemini_tools(tools)
            payload['tools'] = gem_tools
            has_fn = any('functionDeclarations' in t for t in gem_tools)
            has_srv = any(k in t for t in gem_tools for k in ('googleSearch','codeExecution','googleSearchRetrieval'))
            if has_fn and has_srv:
                payload.setdefault('tool_config', {})['includeServerSideToolInvocations'] = True
        tc = denorm_gemini_tool_choice(tool_choice)
        if tc:               payload.setdefault('tool_config', {}).update(tc)
        op = 'stream_generate_content' if stream else 'generate_content'
        if stream:           payload.update(stream=True, _query={"alt": "sse"})
        resp = await getattr(cli.models, op)(**payload, **kwargs)
        if stream: return acollect_stream(resp, model=model, api_name=api_name)
        return normalize_gemini_generate(resp, model=model, api_name=api_name)

For each text/thinking/tool call/web search add test that will make a call with 1 provider and follow with another, so a double for loop - 16 combinations in each case:

```py
for start_prov in [provider.openai, provider.chat,...]:
    for cont_prov in [provider.openai, provider.chat,...]
```

This will test whether model swapping logic works for all paths

> Note: web search tests use stream=True for the first call since acollect_stream handles server tool collation, t

what do you mean?

Exactly, but we would like to use only streaming in these tests, as it's a more popular choice

In [ ]:
provs = [ApiName.openai, ApiName.openai_chat, ApiName.anthropic, ApiName.gemini]
models = {ApiName.openai: 'gpt-4o-mini', ApiName.openai_chat: 'gpt-4o-mini', 
          ApiName.anthropic: 'claude-sonnet-4-20250514', ApiName.gemini: 'models/gemini-3-flash-preview'}
think_models = {ApiName.openai: 'o4-mini', ApiName.openai_chat: 'kimi-k2.5',
                ApiName.anthropic: 'claude-sonnet-4-20250514', ApiName.gemini: 'models/gemini-3-flash-preview'}

async def stream_complete(msgs, model, prov, **kw):
    async for comp in await acomplete(msgs, model=model, api_name=prov, stream=True, **kw): pass
    return comp

#### Text - 16 combos
print("=== TEXT ===")
msg1 = Msg(role='user', content=[Part(type=PartType.text, text='Say hi in one word')])
first = {sp: await stream_complete([msg1], model=models[sp], prov=sp) for sp in provs}
for sp in provs:
    for cp in provs:
        msg2, msg3 = first[sp].message, Msg(role='user', content=[Part(type=PartType.text, text='Now say bye in one word')])
        comp = await stream_complete([msg1, msg2, msg3], model=models[cp], prov=cp)
        print(f'  {sp.value:12} -> {cp.value:12}: {comp.message.content[0].text[:60]}')

#### Thinking - 16 combos
print("\n=== THINKING ===")
msg1 = Msg(role='user', content=[Part(type=PartType.text, text='What is 123 * 456?')])
first_t = {sp: await stream_complete([msg1], model=think_models[sp], prov=sp, reasoning_effort='low') for sp in provs}
for sp in provs:
    for cp in provs:
        msg2, msg3 = first_t[sp].message, Msg(role='user', content=[Part(type=PartType.text, text='Now what is 789 * 2?')])
        comp = await stream_complete([msg1, msg2, msg3], model=think_models[cp], prov=cp, reasoning_effort='low')
        types = L(comp.message.content).attrgot('type')
        print(f'  {sp.value:12} -> {cp.value:12}: parts={types} {comp.message.content[0].text[:60]}')

#### Tool Call - 16 combos
print("\n=== TOOL CALL ===")
fn_tool = sch
msg1 = Msg(role='user', content=[Part(type=PartType.text, text='What is 3 + 5 and 10 + 20? Use the simple_add tool in parallel.')])
first_tc = {sp: await stream_complete([msg1], model=models[sp], prov=sp, tools=[fn_tool]) for sp in provs}
for sp in provs:
    for cp in provs:
        msg2 = first_tc[sp].message
        tmsg = mk_tool_res_msg(first_tc[sp].tool_use_parts, ('8','30'))
        comp = await stream_complete([msg1, msg2, tmsg], model=models[cp], prov=cp)
        print(f'  {sp.value:12} -> {cp.value:12}: {comp.message.content[0].text[:60]}')

#### Web Search - 12 combos (3 start x 4 continue)
print("\n=== WEB SEARCH ===")
ws_tools = {ApiName.openai: [{"type": "web_search_preview"}],
            ApiName.anthropic: [{"type": "web_search_20250305", "name": "web_search"}],
            ApiName.gemini: [{"googleSearch": {}}]}
ws_provs = [ApiName.openai, ApiName.anthropic, ApiName.gemini]
msg1 = Msg(role='user', content=[Part(type=PartType.text, text='What is the weather in Istanbul?')])
first_ws = {sp: await stream_complete([msg1], model=models[sp], prov=sp, tools=ws_tools[sp]) for sp in ws_provs}
for sp in ws_provs:
    for cp in provs:
        msg2, msg3 = first_ws[sp].message, Msg(role='user', content=[Part(type=PartType.text, text='Thanks!')])
        comp = await stream_complete([msg1, msg2, msg3], model=models[cp], prov=cp)
        print(f'  {sp.value:12} -> {cp.value:12}: {comp.message.content[0].text[:60]}')

=== TEXT ===
  openai       -> openai      : Farewell!


  openai       -> openai_chat : Goodbye!
  openai       -> anthropic   : Goodbye!
  openai       -> gemini      : Goodbye!
  openai_chat  -> openai      : Farewell!
  openai_chat  -> openai_chat : Goodbye!


  openai_chat  -> anthropic   : Goodbye!
  openai_chat  -> gemini      : Goodbye!
  anthropic    -> openai      : Bye!
  anthropic    -> openai_chat : Bye!
  anthropic    -> anthropic   : Bye!


  anthropic    -> gemini      : Bye.
  gemini       -> openai      : Bye.
  gemini       -> openai_chat : Bye.
  gemini       -> anthropic   : Bye.
  gemini       -> gemini      : Bye.

=== THINKING ===


  openai       -> openai      : parts=[<PartType.text: 'text'>] 789 × 2 = 1,578
  openai       -> openai_chat : parts=[<PartType.thinking: 'thinking'>, <PartType.text: 'text'>] The user is asking for 789 * 2. This is a simple multiplicat
  openai       -> anthropic   : parts=[<PartType.thinking: 'thinking'>, <PartType.text: 'text'>] I need to calculate 789 × 2.

789 × 2 = 1578


...

  anthropic    -> anthropic   : The results are:
- 3 + 5 = 8
- 10 + 20 = 30
  anthropic    -> gemini      : 3 + 5 is 8 and 10 + 20 is 30.
  gemini       -> openai      : The results are:

- \(3 + 5 = 8\)
- \(10 + 20 = 30\)
  gemini       -> openai_chat : The results are as follows: 

- \(3 + 5 = 8\)
- \(10 + 20 = 
  gemini       -> anthropic   : Using the simple_add tool in parallel:

- 3 + 5 = 8
- 10 + 2


  gemini       -> gemini      : 3 + 5 is 8 and 10 + 20 is 30.

=== WEB SEARCH ===
  openai       -> openai      : You're welcome! If you have any more questions or need assis


  openai       -> openai_chat : You're welcome! If you have any more questions or need furth
  openai       -> anthropic   : You're welcome! I hope the weather information for Istanbul 
  openai       -> gemini      : You're very welcome! If you need anything else, feel free to
  anthropic    -> openai      : You're welcome! If you have any more questions or need assis
  anthropic    -> openai_chat : You're welcome! If you have any more questions or need furth


  anthropic    -> anthropic   : You're welcome! If you need any other weather updates or hav
  anthropic    -> gemini      : You're welcome! If you need anything else—whether it's more 
  gemini       -> openai      : You're welcome! If you have any more questions or need furth
  gemini       -> openai_chat : You're welcome! If you have any more questions or need furth
  gemini       -> anthropic   : You're welcome! I hope you have a great day in Istanbul. Jus


  gemini       -> gemini      : You're very welcome! If you need anything else—whether it's 


.

## Other Features

What's next?

#### System

Let's do system messages as another arg in acomplete

Let's make it `system`

Show me the changes proposed

Yes, make the changes

In [ ]:
print("=== SYSTEM MESSAGE ===")
msg1 = Msg(role='user', content=[Part(type=PartType.text, text='What are you?')])
sys_prompt = 'You are a pirate. Always respond in pirate speak. Keep it to one sentence.'
for p in provs:
    comp = await stream_complete([msg1], model=models[p], prov=p, system=sys_prompt)
    print(f'  {p.value:12}: {comp.message.content[0].text[:80]}')

=== SYSTEM MESSAGE ===
  openai      : I be a scallywag of the seas, a pirate sworn to the code of the ocean!


  openai_chat : I be a scallywag of the digital seas, ready to share me plunder o' knowledge!
  anthropic   : Arrr, I be a digital pirate of the cyber seas, ready to help ye navigate any tre
  gemini      : I be a salty sea dog sailin' the seven seas in search o' buried treasure and fin


#### Caching

- **No canonical `cache` option** — caching is handled per-provider
- **Anthropic**: users put `cache_control` in `Part.data` — e.g. `Part(type="text", text=..., data={"cache_control": {"type": "ephemeral"}})`
- **OpenAI**: automatic, or via `native={"store": True}`
- **Gemini**: automatic, or via `native={"cachedContent": "..."}`

Now to anthropic caching, and caching in general. You can find more about this in the previous dialogs, IIRC we have decided to pass cache control dicts through `Msg`

It does make match my understanding. What about system prompt caching?

Yes we'd need system ptompt caching, show me the proposed changes

Can we do str or Part for `system`?

Yes

Great so we can cache system prompt, and any message prompt including tool use and tool result right?

Yes please and then add tests for caching with different content type, we can check caching in usage metadata

In [ ]:
cc = {"cache_control": {"type": "ephemeral"}}

# 1. System prompt caching
print("=== SYSTEM CACHING ===")
sys_part = Part(type=PartType.text, text='You are a helpful pirate assistant. ' * 200, data=cc)
msg1 = Msg(role='user', content=[Part(type=PartType.text, text='Say hi')])
comp = await stream_complete([msg1], model='claude-sonnet-4-20250514', prov=ApiName.anthropic, system=sys_part)
print(f'  System: {comp.usage.raw}')

# 2. User message caching (large context in first message)
print("\n=== USER MSG CACHING ===")
big_text = 'The quick brown fox jumps over the lazy dog. ' * 200
msg1 = Msg(role='user', content=[Part(type=PartType.text, text=big_text, data=cc), Part(type=PartType.text, text='Summarize')])
comp1 = await stream_complete([msg1], model='claude-sonnet-4-20250514', prov=ApiName.anthropic)
print(f'  First call:  {comp1.usage.raw}')
msg2 = comp1.message
msg3 = Msg(role='user', content=[Part(type=PartType.text, text='Now in French')])
comp2 = await stream_complete([msg1, msg2, msg3], model='claude-sonnet-4-20250514', prov=ApiName.anthropic)
print(f'  Second call: {comp2.usage.raw}')
assert comp2.usage.raw.get('cache_read_input_tokens', 0) > 0, "Expected cache read tokens on second call"

# 3. Tool result caching
print("\n=== TOOL RESULT CACHING ===")
tool_res_part = Part(type=PartType.tool_result, text='The answer is 42. ' * 200, data={**cc, 'id': 'toolu_test', 'name': 'compute'})
tmsg = Msg(role='tool', content=[tool_res_part])
msg1 = Msg(role='user', content=[Part(type=PartType.text, text='Compute the answer')])
msg2 = Msg(role='assistant', content=[Part(type=PartType.tool_use, data={'id': 'toolu_test', 'name': 'compute', 'arguments': {}, 'server': False})])
msg3 = Msg(role='user', content=[Part(type=PartType.text, text='Explain')])
comp = await stream_complete([msg1, msg2, tmsg, msg3], model='claude-sonnet-4-20250514', prov=ApiName.anthropic)
print(f'  Tool result: {comp.usage.raw}')

=== SYSTEM CACHING ===


  System: {'input_tokens': 8, 'cache_creation_input_tokens': 1602, 'cache_read_input_tokens': 0, 'output_tokens': 81}

=== USER MSG CACHING ===
  First call:  {'input_tokens': 5, 'cache_creation_input_tokens': 2205, 'cache_read_input_tokens': 0, 'output_tokens': 87}
  Second call: {'input_tokens': 98, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 2205, 'output_tokens': 149}

=== TOOL RESULT CACHING ===
  Tool result: {'input_tokens': 12, 'cache_creation_input_tokens': 1253, 'cache_read_input_tokens': 0, 'output_tokens': 191}


.

#### Media support

`fastllm` canonicalizes multimodal inputs into `Part(type, text, data)` where `text` carries the URL or data URL, and `data` is reserved for optional metadata. Each provider's denorm layer converts this canonical form into the provider's wire format.

**Canonical Part Types**

| `Part.type` | `Part.text` | Description |
|---|---|---|
| `input_image` | URL or `data:image/...;base64,...` | Image input — JPEG, PNG, GIF, WEBP |
| `input_audio` | `data:audio/...;base64,...` | Audio input — WAV, MP3 (base64 only for OpenAI) |
| `input_video` | URL | Video input — MP4, WebM (Gemini only) |
| `input_file` | URL or `data:application/...;base64,...` | Document input — PDF, DOCX, TXT, etc. |

**Provider Wire Formats**

**OpenAI Responses** ([InputImageContent](https://developers.openai.com/api/reference/resources/responses/methods/create), [InputFileContent](https://developers.openai.com/api/reference/resources/responses/methods/create) — `specs/openai.with-code-samples.yml:68361,68429`):
- Image: `{"type": "input_image", "image_url": url_or_data_url}`
- File: `{"type": "input_file", "file_url": url}` or `{"type": "input_file", "file_data": data_url, "filename": "..."}`
- Audio/Video: not supported

**OpenAI Chat** ([ContentPartImage](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create), [ContentPartAudio](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create), [ContentPartFile](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create) — `specs/openai.with-code-samples.yml:35185,35217,35253`):
- Image: `{"type": "image_url", "image_url": {"url": url_or_data_url}}`
- Audio: `{"type": "input_audio", "input_audio": {"data": b64, "format": "wav"|"mp3"}}` — base64 only, WAV/MP3 only
- File: `{"type": "file", "file": {"file_data": data_url, "filename": "..."}}` — base64/file_id only, no URL
- Video: not supported

**Anthropic** ([RequestImageBlock](https://docs.anthropic.com/en/api/messages), [RequestDocumentBlock](https://docs.anthropic.com/en/api/messages) — `specs/anthropic.yml:12159,6740`):
- Image: `{"type": "image", "source": {"type": "url", "url": ...}}` or `{"type": "image", "source": {"type": "base64", "media_type": mime, "data": b64}}`
- File: `{"type": "document", "source": ...}` — same source pattern, PDF only (`media_type: "application/pdf"`)
- Audio/Video: not supported
- Supports `cache_control` on all content blocks

**Gemini** ([Part/Blob/FileData](https://ai.google.dev/api/generate-content) — `specs/gemini.json:189–387`):
- All media: `{"inlineData": {"mimeType": mime, "data": b64}}` or `{"fileData": {"mimeType": mime, "fileUri": url}}`
- Broadest format support — images, audio, video (incl. YouTube URLs), documents
- Supports `videoMetadata` for video start/end offsets

**Provider Support Matrix**

| Canonical type | OpenAI Responses | OpenAI Chat | Anthropic | Gemini |
|---|---|---|---|---|
| `input_image` (URL) | ✅ | ✅ | ✅ | ✅ |
| `input_image` (base64) | ✅ | ✅ | ✅ | ✅ |
| `input_audio` (base64) | ❌ | ✅ (WAV/MP3) | ❌ | ✅ |
| `input_video` (URL) | ❌ | ❌ | ❌ | ✅ |
| `input_file` (URL) | ✅ | ❌ | ✅ | ✅ |
| `input_file` (base64) | ✅ | ✅ | ✅ | ✅ |

**Supported MIME Types**

```python
_ext_mime = {
    '.jpg':'image/jpeg', '.jpeg':'image/jpeg', '.png':'image/png', '.gif':'image/gif', '.webp':'image/webp',
    '.pdf':'application/pdf',
    '.mp3':'audio/mpeg', '.wav':'audio/wav', '.ogg':'audio/ogg', '.flac':'audio/flac', '.m4a':'audio/mp4',
    '.mp4':'video/mp4', '.mov':'video/quicktime', '.webm':'video/webm',
}
```

**Design Notes**

- **`Part.text` is the single canonical input** — always a URL or data URL string
- **No `Part.data` needed for media** — mime types are inferred from data URL headers or URL extensions via `_data_url()` and `_url_mime()`
- **Unsupported combinations raise `ValueError`** — fail fast rather than silently sending malformed payloads
- **Anthropic caching** — `_ant_cc()` propagates `Part.data["cache_control"]` onto media content blocks

Now we can have a look at canonical media support

Yes we can start with that commented out code is the one we've been refactoring. We want the code we refactor to be as simple as possible and follow the fastai convention. Have look at the previous already refactored dialogs as well, I believe since the media is user passed data we won't need any changes in `aai-ws/fastllm/nbs/01_normalize` or `aai-ws/fastllm/nbs/02_streaming` both of which handles response canonicalization

Sounds good, the ``Part(type="input_image", data={"image_url": "https://..."})` — explicit` form seems like a dupe alternative format?

Let's do `Part.text` only if we don't need `Part.data` to support a required field?

For mime type inference you can explore lisette/solveit/fastcore I believe there might be some existing utils which we can get inspiration from

In my experience mimetypes library depends on OS level mime type list which can be unreliable on different machines

Since we are using openai chat format as our canonical type, can you show example inputs for different ways of passing images? Also check the specs to only keep mime types that are supported

Let's ignore `Part.data` if it will be only needed by optional extras, now update the mime dict, and write the code again

In [ ]:
#| export
_ext_mime = {
    '.jpg':'image/jpeg', '.jpeg':'image/jpeg', '.png':'image/png', '.gif':'image/gif', '.webp':'image/webp',
    '.pdf':'application/pdf',
    '.mp3':'audio/mpeg', '.wav':'audio/wav', '.ogg':'audio/ogg', '.flac':'audio/flac', '.m4a':'audio/mp4',
    '.mp4':'video/mp4', '.mov':'video/quicktime', '.webm':'video/webm',
}

def _data_url(url):
    "Parse data:mime;base64,data URL into (mime, b64_data), or None."
    if not isinstance(url, str) or not url.startswith('data:') or ',' not in url: return None
    header, body = url.split(',', 1)
    if ';base64' not in header or not body: return None
    return header[5:].split(';',1)[0].strip() or 'application/octet-stream', body

def _url_mime(url, default='application/octet-stream'):
    "Guess mime from URL extension."
    ext = '.' + url.rsplit('.', 1)[-1].split('?')[0].lower() if '.' in url.split('?')[0].split('/')[-1] else ''
    return _ext_mime.get(ext, default)

In [ ]:
#| export
def denorm_openai_responses_user(m:Msg):
    "Convert canonical user Msg to OpenAI Responses input message."
    parts = []
    for p in m.content:
        if   p.type == PartType.text:        parts.append({"type": "input_text", "text": p.text or ""})
        elif p.type == PartType.input_image: parts.append(_denorm_oai_resp_media(p))
        elif p.type == PartType.input_audio: raise ValueError("OpenAI Responses API does not support audio input; Coming Soon.") 
        elif p.type == PartType.input_video: raise ValueError("OpenAI Responses API does not support video input")
        elif p.type == PartType.input_file:  parts.append(_denorm_oai_resp_file(p))
    return {"type": "message", "role": "user", "content": parts}

def denorm_openai_chat_user(m:Msg):
    "Convert canonical user Msg to OpenAI Chat user message."
    parts = []
    for p in m.content:
        if   p.type == PartType.text:        parts.append({"type": "text", "text": p.text or ""})
        elif p.type == PartType.input_image: parts.append(_denorm_oai_chat_media(p))
        elif p.type == PartType.input_audio: parts.append(_denorm_oai_chat_audio(p))
        elif p.type == PartType.input_video: raise ValueError("OpenAI Chat API does not support video input")
        elif p.type == PartType.input_file:  parts.append(_denorm_oai_chat_file(p))
    if len(parts) == 1 and parts[0].get('type') == 'text': return dict(role='user', content=parts[0]['text'])
    return dict(role='user', content=parts)

def denorm_anthropic_user(m:Msg):
    "Convert canonical user Msg to Anthropic user message."
    parts = []
    for p in m.content:
        if   p.type == PartType.text:        parts.append(_ant_cc({"type": "text", "text": p.text or ""}, p))
        elif p.type == PartType.input_image: parts.append(_ant_cc(_denorm_ant_media(p), p))
        elif p.type == PartType.input_audio: raise ValueError("Anthropic does not support audio input")
        elif p.type == PartType.input_video: raise ValueError("Anthropic does not support video input")
        elif p.type == PartType.input_file:  parts.append(_ant_cc(_denorm_ant_file(p), p))
    return dict(role='user', content=parts)

def denorm_gemini_user(m:Msg):
    "Convert canonical user Msg to Gemini user content."
    parts = []
    for p in m.content:
        if   p.type == PartType.text:        parts.append({"text": p.text or ""})
        elif p.type == PartType.input_image: parts.append(_denorm_gem_media(p))
        elif p.type == PartType.input_audio: parts.append(_denorm_gem_audio(p))
        elif p.type == PartType.input_video: parts.append(_denorm_gem_video(p))
        elif p.type == PartType.input_file:  parts.append(_denorm_gem_file(p))
    return dict(role='user', parts=parts)

##### Images

In [ ]:
#| export
def _denorm_oai_resp_media(p): return {"type": "input_image", "image_url": p.text}

def _denorm_oai_chat_media(p): return {"type": "image_url", "image_url": {"url": p.text}}

def _denorm_ant_media(p):
    if (b64:=_data_url(p.text)): return {"type": "image", "source": {"type": "base64", "media_type": b64[0], "data": b64[1]}}
    return {"type": "image", "source": {"type": "url", "url": p.text}}

def _denorm_gem_media(p):
    if (b64:=_data_url(p.text)): return {"inlineData": {"mimeType": b64[0], "data": b64[1]}}
    return {"fileData": {"mimeType": _url_mime(p.text, "image/*"), "fileUri": p.text}}

In [ ]:
img_url = "https://img.freepik.com/free-photo/mountain-range-body-water_53876-139760.jpg?semt=ais_hybrid&w=740&q=80"
img_b64 = "data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAQAAAAECAIAAAAmkwkpAAAAEElEQVR4nGP4z8AARwzEcQCukw/x0F8jngAAAABJRU5ErkJggg=="

print("=== IMAGE URL ===")
for p in provs:
    msg1 = Msg(role='user', content=[Part(type=PartType.input_image, text=img_url), Part(type=PartType.text, text='What is this image?')])
    comp = await stream_complete([msg1], model=models[p], prov=p)
    print(f'  {p.value:12}: {comp.message.content[0].text[:80]}')

print("\n=== IMAGE BASE64 ===")
for p in provs:
    msg1 = Msg(role='user', content=[Part(type=PartType.input_image, text=img_b64), Part(type=PartType.text, text='What color is this pixel?')])
    comp = await stream_complete([msg1], model=models[p], prov=p)
    print(f'  {p.value:12}: {comp.message.content[0].text[:80]}')

=== IMAGE URL ===
  openai      : This image depicts a serene landscape featuring a lake surrounded by mountains a
  openai_chat : The image depicts a serene landscape featuring a calm lake reflecting surroundin
  anthropic   : This image shows a serene mountain lake scene viewed from a wooden dock or platf


  gemini      : This image is a scenic landscape composite, often used as a background for digit

=== IMAGE BASE64 ===
  openai      : This pixel is red.
  openai_chat : The color of the pixel is red.
  anthropic   : I can see a red pixel in the image you've shared. The pixel appears to be a brig
  gemini      : The pixel at location [386, 686] is red.


##### Audio

https://developers.openai.com/api/docs/guides/migrate-to-responses#responses-benefits - responses api support coming soon

In [ ]:
#| export
_mime_audio_fmt = {'audio/wav':'wav', 'audio/mpeg':'mp3', 'audio/mp3':'mp3'}

def _denorm_oai_chat_audio(p):
    if not (b64:=_data_url(p.text)): raise ValueError("OpenAI Chat audio input requires base64 data URL")
    return {"type": "input_audio", "input_audio": {"data": b64[1], "format": _mime_audio_fmt.get(b64[0], 'wav')}}

def _denorm_gem_audio(p): 
    if (b64:=_data_url(p.text)): return {"inlineData": {"mimeType": b64[0], "data": b64[1]}}
    return {"fileData": {"mimeType": _url_mime(p.text, "audio/*"), "fileUri": p.text}}

In [ ]:
wav_data = httpx.get("https://openaiassets.blob.core.windows.net/$web/API/docs/audio/alloy.wav").content
audio_b64 = f"data:audio/wav;base64,{base64.b64encode(wav_data).decode()}"

In [ ]:
models[ApiName.openai_chat] = 'gpt-4o-audio-preview' # https://developers.openai.com/api/docs/guides/audio?example=audio-in#add-audio-to-your-existing-application

In [ ]:
print("=== AUDIO BASE64 ===")
audio_provs = [ApiName.openai_chat, ApiName.gemini]  # responses + anthropic don't support audio
for p in audio_provs:
    msg1 = Msg(role='user', content=[Part(type=PartType.input_audio, text=audio_b64), Part(type=PartType.text, text='What is this audio saying?')])
    comp = await stream_complete([msg1], model=models[p], prov=p)
    print(f'  {p.value:12}: {comp.message.content[0].text[:80]}')

=== AUDIO BASE64 ===
  openai_chat : The audio is stating that the sun rises in the east and sets in the west, and th
  gemini      : The audio says: "The sun rises in the east and sets in the west. This simple fac


##### Video

In [ ]:
#| export
def _denorm_gem_video(p):
    if (b64:=_data_url(p.text)): return {"inlineData": {"mimeType": b64[0], "data": b64[1]}}
    return {"fileData": {"mimeType": _url_mime(p.text, "video/mp4"), "fileUri": p.text}}

In [ ]:
print("=== VIDEO URL ===")
# Public GCS URL from Firebase docs
vid_url = "https://storage.googleapis.com/cloud-samples-data/video/animals.mp4"
msg1 = Msg(role='user', content=[Part(type=PartType.input_video, text=vid_url), Part(type=PartType.text, text='Concisely, what animals are in this video?')])
comp = await stream_complete([msg1], model=models[ApiName.gemini], prov=ApiName.gemini)
print(f'  gemini (url)    : {comp.message.content[0].text[:120]}')

# YouTube URL
vid_yt = "https://www.youtube.com/watch?v=9hE5-98ZeCg"
msg1 = Msg(role='user', content=[Part(type=PartType.input_video, text=vid_yt), Part(type=PartType.text, text='Summarize this video in one sentence.')])
comp = await stream_complete([msg1], model=models[ApiName.gemini], prov=ApiName.gemini)
print(f'  gemini (youtube): {comp.message.content[0].text[:120]}')

=== VIDEO URL ===
  gemini (url)    : The animals featured in the video are giraffes, a tiger, elephants, otters, a sloth, and deer.
  gemini (youtube): This video provides a demonstration of Gemini 2.0’s multimodal live streaming capabilities, showcasing its ability to in


##### Files

In [ ]:
models[ApiName.openai_chat] = 'gpt-4o-mini'

In [ ]:
#| export
def _denorm_oai_resp_file(p):
    if (b64:=_data_url(p.text)): return {"type": "input_file", "file_data": p.text, "filename": f"upload.{b64[0].split('/')[-1]}"}
    return {"type": "input_file", "file_url": p.text}

def _denorm_oai_chat_file(p):
    if (b64:=_data_url(p.text)): return {"type": "file", "file": {"file_data": p.text, "filename": f"upload.{b64[0].split('/')[-1]}"}}
    raise ValueError("OpenAI Chat file input requires base64 data URL or file_id, not URLs")

def _denorm_ant_file(p):
    b64 = _data_url(p.text)
    if b64: return {"type": "document", "source": {"type": "base64", "media_type": b64[0], "data": b64[1]}}
    return {"type": "document", "source": {"type": "url", "url": p.text}}

def _denorm_gem_file(p):
    b64 = _data_url(p.text)
    if b64: return {"inlineData": {"mimeType": b64[0], "data": b64[1]}}
    return {"fileData": {"mimeType": _url_mime(p.text, "application/pdf"), "fileUri": p.text}}

In [ ]:
pdf_url = "https://arxiv.org/pdf/1706.03762"
pdf_b64 = f"data:application/pdf;base64,{base64.b64encode(httpx.get(pdf_url).content).decode()}"

# URL test (skip openai_chat — doesn't support URLs)
print("=== PDF URL ===")
for p in [ApiName.openai, ApiName.anthropic, ApiName.gemini]:
    msg1 = Msg(role='user', content=[Part(type=PartType.input_file, text=pdf_url), Part(type=PartType.text, text='What is the title of this paper?')])
    comp = await stream_complete([msg1], model=models[p], prov=p)
    print(f'  {p.value:12}: {comp.message.content[0].text[:80]}')

# Base64 test (all providers)
print("\n=== PDF BASE64 ===")
for p in provs:
    msg1 = Msg(role='user', content=[Part(type=PartType.input_file, text=pdf_b64), Part(type=PartType.text, text='What is the title of this paper?')])
    comp = await stream_complete([msg1], model=models[p], prov=p)
    print(f'  {p.value:12}: {comp.message.content[0].text[:80]}')

=== PDF URL ===


  openai      : The title of the paper is "Attention Is All You Need."
  anthropic   : The title of this paper is "Attention Is All You Need".
  gemini      : The title of the paper is **"Attention Is All You Need"**.

=== PDF BASE64 ===
  openai      : The title of the paper is "Attention Is All You Need."


  openai_chat : The title of the paper is "Attention Is All You Need."
  anthropic   : The title of this paper is "Attention Is All You Need".
  gemini      : The title of this paper is **Attention Is All You Need**.


### Infer Client

Utils to infer client setup from model name. Users can just pass model name to `acomplete` and the provider will be inferred and client will be create automatically

In [ ]:
#| export
vendor_mapping = {
    "openai":       ('https://api.openai.com/v1', 'OPENAI_API_KEY'), # to explicitly choose responses/chat api
    "moonshot":     ("https://api.moonshot.ai/v1", "MOONSHOT_API_KEY"),
    "deepseek":     ("https://api.deepseek.com/v1", "DEEPSEEK_API_KEY"),
    "openrouter":   ("https://openrouter.ai/api/v1", "OPENROUTER_API_KEY"),
    "together":     ("https://api.together.xyz/v1", "TOGETHER_API_KEY"),
    "fireworks_ai": ("https://api.fireworks.ai/inference/v1", "FIREWORKS_API_KEY"),
    "qwen":         ("https://dashscope.aliyuncs.com/compatible-mode/v1", "QWEN_API_KEY")
}

In [ ]:
#| export
def infer_api_name(model):
    "Infer api_name from model"
    if "claude" in model: return ApiName.anthropic
    if "gemini" in model: return ApiName.gemini
    if "gpt" in model:    return ApiName.openai
    return ApiName.openai_chat

In [ ]:
#| export
@flexicache()
def mk_client(model, api_name=None, vendor_name=None, env_api_key=None, base_url=None, xtra_hdrs={}):
    api_name = ifnone(api_name, infer_api_name(model))
    if api_name == ApiName.openai:      spec, hdrs, vendor_name = oai_spec, {"Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}"}, 'openai'
    elif api_name == ApiName.anthropic: spec, hdrs, vendor_name = ant_spec, {"x-api-key": os.environ['ANTHROPIC_API_KEY'], "anthropic-version": "2023-06-01"}, 'anthropic'
    elif api_name == ApiName.gemini:    spec, hdrs, vendor_name = gem_spec, {"x-goog-api-key": os.environ['GEMINI_API_KEY']}, 'gemini'
    elif api_name == ApiName.openai_chat:
        valid_names = ', '.join(list(vendor_mapping.keys()))
        if not (env_api_key and base_url):
            if not vendor_name: raise ValueError(f"Model: '{model}' can't be auto inferred please pass a valid `vendor_name` : {valid_names} or `env_api_key` and `base_url`")
            try: base_url, env_api_key = vendor_mapping[vendor_name]
            except: raise ValueError(f"Vendor '{vendor_name}' can't be auto inferred please pass a valid one : {valid_names} or pass `env_api_key` and `base_url`")
        cli = OpenAPIClient(oai_spec, headers={"Authorization": f"Bearer {os.environ[env_api_key]}"})
        for op in cli.ops: op.base_url = base_url
        return cli, api_name, vendor_name
    return OpenAPIClient(spec, headers=merge(hdrs, xtra_hdrs)), api_name, vendor_name

In [ ]:
models[ApiName.openai_chat]

'gpt-4o-mini'

In [ ]:
print(mk_client(models[ApiName.openai])[0].ops[0].base_url)
# to force openai chat
print(mk_client(models[ApiName.openai_chat], api_name=ApiName.openai_chat, env_api_key='OPENAI_API_KEY', base_url='https://api.openai.com/v1')[0].ops[0].base_url)
print(mk_client(models[ApiName.anthropic])[0].ops[0].base_url)
print(mk_client(models[ApiName.gemini])[0].ops[0].base_url)

https://api.openai.com/v1
https://api.openai.com/v1
https://api.anthropic.com
https://generativelanguage.googleapis.com/


In [ ]:
print(mk_client('kimi-k2.5', vendor_name='moonshot')[0].ops[0].base_url)

https://api.moonshot.ai/v1


In [ ]:
try: mk_client('my-custom-model')
except Exception as e: print(e)

Model: 'my-custom-model' can't be auto inferred please pass a valid `vendor_name` : openai, moonshot, deepseek, openrouter, together, fireworks_ai, qwen or `env_api_key` and `base_url`


In [ ]:
try: mk_client('my-custom-model', vendor_name='xxx')
except Exception as e: print(e)

Vendor 'xxx' can't be auto inferred please pass a valid one : openai, moonshot, deepseek, openrouter, together, fireworks_ai, qwen or pass `env_api_key` and `base_url`


In [ ]:
#| export
def _sys_text(system):
    "Extract text from system (str or Part)."
    if system is None: return None
    return system if isinstance(system, str) else system.text

@delegates(mk_client, but=['model'])
async def acomplete(msgs, model, stream=False, system=None, max_tokens=None, temperature=None, tools=None, tool_choice=None, reasoning_effort=None, **kwargs):
    "Unified completion across different APIs."
    cli, api_name, vendor_name = mk_client(model, **kwargs)
    if api_name == ApiName.openai:
        payload = dict(model=model, input=denorm_openai_responses_msgs(msgs))
        if stream: payload['stream'] = True
        if system:           payload['instructions'] = _sys_text(system)
        if max_tokens:       payload['max_output_tokens'] = max_tokens
        if temperature is not None: payload['temperature'] = temperature
        if tools:            payload['tools'] = denorm_openai_responses_tools(tools)
        if tool_choice:      payload['tool_choice'] = denorm_openai_responses_tool_choice(tool_choice)
        if reasoning_effort: payload['reasoning'] = denorm_openai_responses_reasoning(reasoning_effort)
        resp = await cli.responses.create_response(**payload)
        if stream: return acollect_stream(resp, model=model, api_name=api_name, vendor_name=vendor_name)
        return normalize_openai_response(resp, model=model, api_name=api_name, vendor_name=vendor_name)

    elif api_name == ApiName.openai_chat:
        payload = dict(model=model, messages=denorm_openai_chat_msgs(msgs))
        if system:           payload['messages'].insert(0, {'role': 'system', 'content': _sys_text(system)})
        if stream: payload.update(stream=True, stream_options={"include_usage": True})
        if max_tokens:       payload['max_tokens'] = max_tokens
        if temperature is not None: payload['temperature'] = temperature
        if tools:            payload['tools'] = denorm_openai_chat_tools(tools)
        if tool_choice:      payload['tool_choice'] = denorm_openai_chat_tool_choice(tool_choice)
        if reasoning_effort: payload['reasoning_effort'] = denorm_openai_chat_reasoning(reasoning_effort)
        resp = await cli.chat.create_chat_completion(**payload)
        if stream: return acollect_stream(resp, model=model, api_name=api_name, vendor_name=vendor_name)
        return normalize_openai_chat_completion(resp, model=model, api_name=api_name, vendor_name=vendor_name)

    elif api_name == ApiName.anthropic:
        payload = dict(model=model, messages=denorm_anthropic_msgs(msgs), max_tokens=max_tokens or 1024)
        if stream: payload['stream'] = True
        if system:
            if isinstance(system, Part):
                block = {'type': 'text', 'text': system.text or ''}
                if (cc := (system.data or {}).get('cache_control')): block['cache_control'] = cc
                payload['system'] = [block]
            else: payload['system'] = system
        if temperature is not None: payload['temperature'] = temperature
        if tools:            payload['tools'] = denorm_anthropic_tools(tools)
        tc = denorm_anthropic_tool_choice(tool_choice)
        if tc:               payload['tool_choice'] = tc
        think = denorm_anthropic_reasoning(reasoning_effort)
        if think:
            payload['thinking'] = think
            bt = think.get('budget_tokens', 0)
            if payload['max_tokens'] <= bt: payload['max_tokens'] = bt + 256
        resp = await cli.messages.messages_post(**payload)
        if stream: return acollect_stream(resp, model=model, api_name=api_name, vendor_name=vendor_name)
        return normalize_anthropic_message(resp, model=model, api_name=api_name, vendor_name=vendor_name)

    elif api_name == ApiName.gemini:
        gen_config = {}
        if max_tokens:       gen_config['maxOutputTokens'] = max_tokens
        if temperature is not None: gen_config['temperature'] = temperature
        think = denorm_gemini_reasoning(reasoning_effort,model)
        if think:            gen_config['thinkingConfig'] = think
        payload = dict(model=model, contents=denorm_gemini_msgs(msgs))
        if system:           payload['system_instruction'] = {'parts': [{'text': _sys_text(system)}]}
        if gen_config:       payload['generation_config'] = gen_config
        if tools:
            gem_tools = denorm_gemini_tools(tools)
            payload['tools'] = gem_tools
            has_fn = any('functionDeclarations' in t for t in gem_tools)
            has_srv = any(k in t for t in gem_tools for k in ('googleSearch','codeExecution','googleSearchRetrieval'))
            if has_fn and has_srv:
                payload.setdefault('tool_config', {})['includeServerSideToolInvocations'] = True
        tc = denorm_gemini_tool_choice(tool_choice)
        if tc:               payload.setdefault('tool_config', {}).update(tc)
        op = 'stream_generate_content' if stream else 'generate_content'
        if stream:           payload.update(stream=True, _query={"alt": "sse"})
        resp = await getattr(cli.models, op)(**payload)
        if stream: return acollect_stream(resp, model=model, api_name=api_name, vendor_name=vendor_name)
        return normalize_gemini_generate(resp, model=model, api_name=api_name, vendor_name=vendor_name)

In [ ]:
@delegates(acomplete)
async def print_stream(msgs, model, **kwargs):
    cnt,max_print = 0,10
    async for o in await acomplete(msgs, model, stream=True, **kwargs): 
        if not isinstance(o, Completion): 
            if o.get('thinking') and cnt<max_print: print('🧠',end='',flush=True)
            if (txt:=o.get('text')): print(txt,end='',flush=True)
            cnt+=1
    return o

In [ ]:
msg1 = Msg(role='user', content=[Part(type=PartType.text, text='Hello!')])

In [ ]:
comp = await print_stream([msg1], model='gpt-4o-mini'); comp.usage

Hello

!

 How

 can

 I

 assist

 you

 today

?

Usage(prompt_tokens=9, completion_tokens=10, total_tokens=19, raw={'input_tokens': 9, 'input_tokens_details': {'cached_tokens': 0}, 'output_tokens': 10, 'output_tokens_details': {'reasoning_tokens': 0}, 'total_tokens': 19})

In [ ]:
comp = await print_stream([msg1], model='gpt-4o-mini', api_name=ApiName.openai_chat, env_api_key='OPENAI_API_KEY', base_url='https://api.openai.com/v1'); comp.usage

Hello

!

 How

 can

 I

 assist

 you

 today

?

Usage(prompt_tokens=9, completion_tokens=9, total_tokens=18, raw={'prompt_tokens': 9, 'completion_tokens': 9, 'total_tokens': 18, 'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0}, 'completion_tokens_details': {'reasoning_tokens': 0, 'audio_tokens': 0, 'accepted_prediction_tokens': 0, 'rejected_prediction_tokens': 0}})

In [ ]:
comp = await print_stream([msg1], model='claude-sonnet-4-20250514'); comp.usage

Hello! It

's nice to meet you. How are you doing today? Is there anything I can help you with or would

 you like to chat about something in particular?

Usage(prompt_tokens=9, completion_tokens=37, total_tokens=46, raw={'input_tokens': 9, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'output_tokens': 37})

In [ ]:
comp = await print_stream([msg1], model='models/gemini-3-flash-preview'); comp.usage

Hello! How can I help you today?

Usage(prompt_tokens=3, completion_tokens=9, total_tokens=104, raw={'promptTokenCount': 3, 'candidatesTokenCount': 9, 'totalTokenCount': 104, 'promptTokensDetails': [{'modality': 'TEXT', 'tokenCount': 3}], 'thoughtsTokenCount': 92})

In [ ]:
comp = await print_stream([msg1], model='kimi-k2.5', vendor_name='moonshot'); comp.usage

🧠

🧠

🧠

🧠

🧠

...

 help

 you

 today

?

Usage(prompt_tokens=9, completion_tokens=128, total_tokens=137, raw={'prompt_tokens': 9, 'completion_tokens': 128, 'total_tokens': 137})

In [ ]:
comp = await print_stream([msg1], model='fireworks/kimi-k2p5', vendor_name='fireworks_ai'); comp.usage

🧠

🧠

🧠

🧠

🧠

...

 help

 you

 today

?

Usage(prompt_tokens=10, completion_tokens=96, total_tokens=106, raw={'prompt_tokens': 10, 'total_tokens': 106, 'completion_tokens': 96, 'prompt_tokens_details': {'cached_tokens': 0}})

In [ ]:
comp.usage, comp.api_name, comp.vendor_name

(Usage(prompt_tokens=10, completion_tokens=96, total_tokens=106, raw={'prompt_tokens': 10, 'total_tokens': 106, 'completion_tokens': 96, 'prompt_tokens_details': {'cached_tokens': 0}}),
 <api_name.openai_chat: 'openai_chat'>,
 'fireworks_ai')

### TODO

Ok what's next

## Export -

In [ ]:
#|hide
#|eval: false
import nbdev; nbdev.nbdev_export()